In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition"
    / "biohub-cell-tracking-during-development/train"
)

CV_PATH = PROJECT / "configs/cv_manifest.csv"
SPLITS_PATH = PROJECT / "configs/dataset_splits.json"

cv = pd.read_csv(CV_PATH)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

print("CV rows:", len(cv))
print("Fold counts:")
print(cv["fold"].value_counts().sort_index())

fold0 = cv[cv["fold"] == 0].copy()

print()
print("Fold 0 validation datasets:", len(fold0))
print(fold0["prefix"].value_counts())
print()
print(fold0.head())

In [ ]:
from pathlib import Path

import pandas as pd
import tracksdata as td

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

TRAIN_DIR = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)

CV_PATH = PROJECT / "configs" / "cv_manifest.csv"

cv = pd.read_csv(CV_PATH)
fold0 = cv.loc[cv["fold"] == 0].copy()

print("tracksdata:", td.__version__)
print("Fold 0 datasets:", len(fold0))
print("Setup: OK")

In [ ]:
rows = []

for _, r in fold0.iterrows():
    name = r["dataset"]

    zarr_path = TRAIN_DIR / f"{name}.zarr"
    geff_path = TRAIN_DIR / f"{name}.geff"

    assert zarr_path.exists(), zarr_path
    assert geff_path.exists(), geff_path

    graph, metadata = td.graph.IndexedRXGraph.from_geff(geff_path)

    estimated_nodes = metadata.extra.get(
        "estimated_number_of_nodes"
    )

    rows.append({
        "dataset": name,
        "prefix": r["prefix"],
        "zarr_exists": zarr_path.exists(),
        "geff_exists": geff_path.exists(),
        "manifest_estimated_nodes": int(r["estimated_nodes"]),
        "geff_estimated_nodes": int(estimated_nodes),
        "estimated_nodes_match": (
            int(r["estimated_nodes"])
            == int(estimated_nodes)
        ),

        # 注意这里有 ()
        "gt_nodes": int(graph.num_nodes()),
        "gt_edges": int(graph.num_edges()),
    })

eval_inputs = pd.DataFrame(rows)

print("Datasets:", len(eval_inputs))
print(
    "Estimated-node metadata matches:",
    eval_inputs["estimated_nodes_match"].sum(),
    "/",
    len(eval_inputs),
)

print("\nDtypes:")
print(
    eval_inputs[
        ["gt_nodes", "gt_edges", "geff_estimated_nodes"]
    ].dtypes
)

print("\nPrefix summary:")
print(
    eval_inputs.groupby("prefix")[
        ["gt_nodes", "gt_edges", "geff_estimated_nodes"]
    ].agg(["mean", "median"])
)

assert len(eval_inputs) == 40
assert eval_inputs["zarr_exists"].all()
assert eval_inputs["geff_exists"].all()
assert eval_inputs["estimated_nodes_match"].all()

print()
print("Fold 0 evaluation inputs: OK")

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import tracksdata as td

OFFICIAL = PROJECT / "external/official"

for p in [
    OFFICIAL / "src",
    OFFICIAL / "scripts",
]:
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

print("Official metric imports: OK")

In [ ]:
def load_graph(path):
    result = td.graph.IndexedRXGraph.from_geff(path)
    return result[0] if isinstance(result, tuple) else result


def evaluate_fold_predictions(
    pred_dir,
    fold_df,
    gt_dir=TRAIN_DIR,
    max_distance=7.0,
):
    pred_dir = Path(pred_dir)

    rows = []
    missing = []
    failed = []

    for _, r in fold_df.iterrows():
        name = r["dataset"]

        pred_path = pred_dir / f"{name}.geff"
        gt_path = gt_dir / f"{name}.geff"

        if not pred_path.exists():
            missing.append(name)
            continue

        try:
            pred_graph = load_graph(pred_path)
            gt_graph = load_graph(gt_path)

            # Official physical voxel scale
            scale = open_dataset(
                gt_dir / name,
                load_image=False,
            ).scale

            er = official_evaluate(
                pred_graph,
                gt_graph,
                scale=scale,
                max_distance=max_distance,
            )

            recall = (
                node_recall(pred_graph, gt_graph)
                if (
                    pred_graph.num_nodes() > 0
                    and pred_graph.num_edges() > 0
                )
                else 0.0
            )

            n_total = float(r["estimated_nodes"])

            metrics = per_sample_metrics(
                er,
                n_total=n_total,
                node_recall=recall,
            )

            rows.append({
                "dataset": name,
                "prefix": r["prefix"],
                "fold": int(r["fold"]),
                "estimated_nodes": int(n_total),
                **metrics,
            })

        except Exception as exc:
            failed.append(
                (name, type(exc).__name__, str(exc))
            )

    result_df = pd.DataFrame(rows)

    overall = (
        summarise(result_df.to_dict("records"))
        if len(result_df)
        else None
    )

    prefix_summary = {}

    if len(result_df):
        for prefix, g in result_df.groupby("prefix"):
            prefix_summary[prefix] = summarise(
                g.to_dict("records")
            )

    return {
        "per_dataset": result_df,
        "overall": overall,
        "by_prefix": prefix_summary,
        "missing": missing,
        "failed": failed,
    }


print("Fold evaluation wrapper: READY")

In [ ]:
import tempfile
import shutil
from pathlib import Path

sanity_name = fold0.iloc[0]["dataset"]

sanity_dir = Path(
    tempfile.mkdtemp(prefix="biohub_metric_sanity_")
)

src = TRAIN_DIR / f"{sanity_name}.geff"
dst = sanity_dir / f"{sanity_name}.geff"

assert src.is_dir(), src

shutil.copytree(src, dst)

sanity_df = fold0[
    fold0["dataset"] == sanity_name
].copy()

sanity = evaluate_fold_predictions(
    sanity_dir,
    sanity_df,
)

print("Dataset:", sanity_name)
print("Overall:")
print(sanity["overall"])

print()
print("Per-dataset:")
print(
    sanity["per_dataset"][
        [
            "dataset",
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "division_tp",
            "division_fp",
            "division_fn",
        ]
    ]
)

In [ ]:
division_candidates = fold0[
    fold0["n_divisions"] > 0
].sort_values(
    "n_divisions",
    ascending=False,
)

print(
    division_candidates[
        ["dataset", "prefix", "n_divisions"]
    ].head(10)
)

In [ ]:
sanity_name = division_candidates.iloc[0]["dataset"]

sanity_dir_div = Path(
    tempfile.mkdtemp(prefix="biohub_metric_div_sanity_")
)

src = TRAIN_DIR / f"{sanity_name}.geff"
dst = sanity_dir_div / f"{sanity_name}.geff"

shutil.copytree(src, dst)

sanity_df_div = fold0[
    fold0["dataset"] == sanity_name
].copy()

sanity_div = evaluate_fold_predictions(
    sanity_dir_div,
    sanity_df_div,
)

print("Dataset:", sanity_name)
print("GT divisions:", sanity_df_div.iloc[0]["n_divisions"])
print()
print("Overall:")
print(sanity_div["overall"])

print()
print(
    sanity_div["per_dataset"][
        [
            "dataset",
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "division_tp",
            "division_fp",
            "division_fn",
        ]
    ]
)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

TRAIN_DIR = (
    PROJECT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
    / "train"
)

CV_PATH = PROJECT / "configs" / "cv_manifest.csv"

PRED_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0"
    / "split_0"
)

cv = pd.read_csv(CV_PATH)
fold0 = cv.loc[cv["fold"] == 0].copy()

name = "44b6_18ced818"
one_dataset = fold0.loc[fold0["dataset"] == name].copy()

print("Prediction exists:", (PRED_DIR / f"{name}.geff").exists())
print("Dataset rows:", len(one_dataset))

In [ ]:
per_dataset_1, overall_1, by_prefix_1, missing_1, failed_1 = (
    evaluate_fold_predictions(
        pred_dir=PRED_DIR,
        fold_df=one_dataset,
        gt_dir=TRAIN_DIR,
    )
)

print("=== Overall ===")
print(overall_1)

print("\n=== Per dataset ===")
display(per_dataset_1)

print("\nMissing:", missing_1)
print("Failed:", failed_1)

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
    / "biohub-cell-tracking-during-development/train"
)

PRED_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0"
    / "split_0"
)

# Official locked source
OFFICIAL_SCRIPTS = PROJECT / "external" / "official" / "scripts"
OFFICIAL_SRC = PROJECT / "external" / "official" / "src"

for p in [OFFICIAL_SCRIPTS, OFFICIAL_SRC]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from evaluate import evaluate_pairs
from tracking_cellmot.metrics import summarise

print("Predicted GEFFs:", len(list(PRED_DIR.glob("*.geff"))))

rows, skipped = evaluate_pairs(
    pred_dir=PRED_DIR,
    gt_dir=TRAIN_DIR,
    max_distance=7.0,
)

summary = summarise(rows)

print("\n=== OFFICIAL METRIC SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n=== PER SAMPLE ===")
display(pd.DataFrame(rows))

print("\nSkipped:", skipped)

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
    / "biohub-cell-tracking-during-development/train"
)

PRED_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0"
    / "split_0"
)

OFFICIAL_SCRIPTS = PROJECT / "external" / "official" / "scripts"
OFFICIAL_SRC = PROJECT / "external" / "official" / "src"

for p in [OFFICIAL_SCRIPTS, OFFICIAL_SRC]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from evaluate import evaluate_pairs
from tracking_cellmot.metrics import summarise

pred_files = sorted(PRED_DIR.glob("*.geff"))
print("Predicted GEFFs:", len(pred_files))
assert len(pred_files) == 40

rows, skipped = evaluate_pairs(
    pred_dir=PRED_DIR,
    gt_dir=TRAIN_DIR,
    max_distance=7.0,
)

summary = summarise(rows)

print("\n=== FOLD 0 OFFICIAL BASELINE ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSkipped:", skipped)

assert len(rows) == 40
assert len(skipped) == 0

print("\nFold 0 official evaluation: OK")

In [ ]:
names = sorted(p.stem for p in PRED_DIR.glob("*.geff"))

assert len(names) == len(rows) == 40

baseline_df = pd.DataFrame(rows)
baseline_df.insert(0, "dataset", names)

cv_meta = pd.read_csv(PROJECT / "configs" / "cv_manifest.csv")[
    ["dataset", "prefix", "estimated_nodes"]
]

baseline_df = baseline_df.merge(
    cv_meta,
    on="dataset",
    how="left",
    validate="one_to_one",
)

print("=== BY PREFIX ===")

for prefix, g in baseline_df.groupby("prefix"):
    s = summarise(g.to_dict("records"))

    print(f"\n{prefix}")
    print(f"n                    = {s['n']}")
    print(f"edge_jaccard         = {s['edge_jaccard']:.6f}")
    print(f"adj_edge_jaccard     = {s['adj_edge_jaccard']:.6f}")
    print(f"division_jaccard     = {s['division_jaccard']:.6f}")
    print(f"node_recall          = {s['node_recall']:.6f}")
    print(f"score                = {s['score']:.6f}")
    print(
        "division TP/FP/FN     = "
        f"{s['division_tp']}/{s['division_fp']}/{s['division_fn']}"
    )

print("\n=== NODE COUNT RATIO BY PREFIX ===")
display(
    baseline_df.groupby("prefix")["total_node_ratio"]
    .agg(["mean", "median", "min", "max"])
)

In [ ]:
import tracksdata as td

sample_path = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0"
    / "split_0"
    / "6bba_df673a83.geff"
)

loaded = td.graph.IndexedRXGraph.from_geff(sample_path)
graph = loaded[0] if isinstance(loaded, tuple) else loaded

print("Graph type:", type(graph))
print("Nodes:", graph.num_nodes())
print("Edges:", graph.num_edges())

print("\n=== Edge-related API ===")
for name in dir(graph):
    if "edge" in name.lower():
        print(name)

In [ ]:
import inspect

methods = [
    "edge_attr_keys",
    "edge_ids",
    "edge_list",
    "edges",
    "edge_attrs",
]

for name in methods:
    obj = getattr(graph, name)

    print(f"\n=== {name} ===")

    try:
        print("signature:", inspect.signature(obj))
    except Exception as e:
        print("signature unavailable:", e)

    doc = inspect.getdoc(obj)
    if doc:
        print("doc:")
        print(doc[:800])

print("\n=== Edge attribute keys ===")
try:
    print(graph.edge_attr_keys())
except Exception as e:
    print("ERROR:", type(e).__name__, e)

In [ ]:
import inspect

for name in ["edge_attrs", "bulk_remove_edges"]:
    obj = getattr(graph, name)

    print(f"\n=== {name} ===")

    try:
        print("signature:", inspect.signature(obj))
    except Exception as e:
        print("signature unavailable:", e)

    doc = inspect.getdoc(obj)
    if doc:
        print(doc[:1200])

# 顺便看前 5 条真实 edge 的 ID 和 source/target
print("\n=== First 5 edges ===")
edge_ids = graph.edge_ids()
edge_list = graph.edge_list()

for eid, pair in list(zip(edge_ids, edge_list))[:5]:
    print("edge_id:", eid, "source/target:", pair)

In [ ]:
attrs = graph.edge_attrs(
    attr_keys=["edge_prob", "edge_dist"],
    unpack=True,
)

print(attrs.head())
print("\nColumns:", attrs.columns)
print("Rows:", attrs.height)

print("\nEdge count:", graph.num_edges())
print("Edge IDs:", len(graph.edge_ids()))
print("Edge list:", len(graph.edge_list()))

assert attrs.height == graph.num_edges()

print("\nAlignment check: OK")

In [ ]:
from pathlib import Path
import shutil
import tracksdata as td

from tracking_cellmot.io import save_graph

BASE_PRED_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0"
    / "split_0"
)

NO_DIV_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_no_division"
    / "split_0"
)

# 清理旧的 ablation 输出；不会碰 baseline
if NO_DIV_DIR.exists():
    shutil.rmtree(NO_DIV_DIR)
NO_DIV_DIR.mkdir(parents=True, exist_ok=True)

stats = []

for src_path in sorted(BASE_PRED_DIR.glob("*.geff")):
    loaded = td.graph.IndexedRXGraph.from_geff(src_path)
    graph = loaded[0] if isinstance(loaded, tuple) else loaded

    before_edges = graph.num_edges()

    edge_df = graph.edge_attrs(
        attr_keys=["edge_prob"],
        unpack=True,
    )

    # 对每个 source，只保留 edge_prob 最大的一条 outgoing edge
    # 概率完全相同时，用更小 edge_id 做 deterministic tie-break
    best_by_source = {}
    remove_ids = []

    for row in edge_df.iter_rows(named=True):
        source = int(row["source_id"])
        edge_id = int(row["edge_id"])
        prob = float(row["edge_prob"])

        candidate = (prob, -edge_id)

        if source not in best_by_source:
            best_by_source[source] = (candidate, edge_id)
        else:
            current_candidate, current_edge_id = best_by_source[source]

            if candidate > current_candidate:
                remove_ids.append(current_edge_id)
                best_by_source[source] = (candidate, edge_id)
            else:
                remove_ids.append(edge_id)

    if remove_ids:
        graph.bulk_remove_edges(remove_ids)

    after_edges = graph.num_edges()

    # 验证每个 parent 最多 1 个 child
    check_df = graph.edge_attrs(
        attr_keys=["edge_prob"],
        unpack=True,
    )

    if check_df.height:
        max_outdegree = (
            check_df
            .group_by("source_id")
            .len()
            ["len"]
            .max()
        )
    else:
        max_outdegree = 0

    assert max_outdegree <= 1

    out_path = NO_DIV_DIR / src_path.name
    save_graph(graph, out_path)

    stats.append({
        "dataset": src_path.stem,
        "edges_before": before_edges,
        "edges_after": after_edges,
        "edges_removed": len(remove_ids),
        "max_outdegree": max_outdegree,
    })

print("Generated:", len(stats), "datasets")
print("Output:", NO_DIV_DIR)

ablation_stats = pd.DataFrame(stats)

print("\n=== Ablation summary ===")
display(
    ablation_stats[
        ["edges_before", "edges_after", "edges_removed"]
    ].sum().to_frame("total").T
)

print(
    "\nDatasets with removed fork edges:",
    (ablation_stats["edges_removed"] > 0).sum(),
    "/",
    len(ablation_stats),
)

print(
    "Maximum remaining outdegree:",
    ablation_stats["max_outdegree"].max(),
)

assert len(list(NO_DIV_DIR.glob("*.geff"))) == 40
print("\nNo-division GEFF generation: OK")

In [ ]:
# Evaluate no-division ablation with the locked official metric

no_div_rows, no_div_skipped = evaluate_pairs(
    pred_dir=NO_DIV_DIR,
    gt_dir=TRAIN_DIR,
    max_distance=7.0,
)

no_div_summary = summarise(no_div_rows)

BASELINE_SCORE = 0.6964058797724787
BASELINE_EDGE = 0.7172921663344931
BASELINE_ADJ_EDGE = 0.6957479850356366
BASELINE_DIV = 0.006578947368421052

print("\n=== NO-DIVISION ABLATION ===")
for k, v in no_div_summary.items():
    print(f"{k}: {v}")

print("\n=== COMPARISON ===")
print(
    f"Score:          {BASELINE_SCORE:.6f} -> "
    f"{no_div_summary['score']:.6f}  "
    f"({no_div_summary['score'] - BASELINE_SCORE:+.6f})"
)

print(
    f"Edge Jaccard:   {BASELINE_EDGE:.6f} -> "
    f"{no_div_summary['edge_jaccard']:.6f}  "
    f"({no_div_summary['edge_jaccard'] - BASELINE_EDGE:+.6f})"
)

print(
    f"Adj Edge Jacc:  {BASELINE_ADJ_EDGE:.6f} -> "
    f"{no_div_summary['adj_edge_jaccard']:.6f}  "
    f"({no_div_summary['adj_edge_jaccard'] - BASELINE_ADJ_EDGE:+.6f})"
)

print(
    f"Division Jacc:  {BASELINE_DIV:.6f} -> "
    f"{no_div_summary['division_jaccard']:.6f}"
)

print(
    "Division TP/FP/FN:",
    no_div_summary["division_tp"],
    no_div_summary["division_fp"],
    no_div_summary["division_fn"],
)

print("\nSkipped:", no_div_skipped)

assert len(no_div_rows) == 40
assert len(no_div_skipped) == 0

In [ ]:
import numpy as np
import pandas as pd
import tracksdata as td

fork_rows = []

for pred_path in sorted(BASE_PRED_DIR.glob("*.geff")):
    loaded = td.graph.IndexedRXGraph.from_geff(pred_path)
    g = loaded[0] if isinstance(loaded, tuple) else loaded

    edf = g.edge_attrs(
        attr_keys=["edge_prob", "edge_dist"],
        unpack=True,
    ).to_pandas()

    # 只看有 >=2 outgoing edges 的 parent
    for source_id, group in edf.groupby("source_id"):
        if len(group) < 2:
            continue

        group = group.sort_values(
            ["edge_prob", "edge_id"],
            ascending=[False, True],
        )

        e1 = group.iloc[0]
        e2 = group.iloc[1]

        fork_rows.append({
            "dataset": pred_path.stem,
            "prefix": pred_path.stem.split("_")[0],
            "source_id": int(source_id),

            "p1": float(e1["edge_prob"]),
            "p2": float(e2["edge_prob"]),
            "prob_margin": float(e1["edge_prob"] - e2["edge_prob"]),
            "prob_ratio": float(e2["edge_prob"] / max(e1["edge_prob"], 1e-12)),

            "dist1": float(e1["edge_dist"]),
            "dist2": float(e2["edge_dist"]),
        })

fork_df = pd.DataFrame(fork_rows)

print("Fork parents:", len(fork_df))

print("\n=== Probability summary ===")
display(
    fork_df[
        ["p1", "p2", "prob_margin", "prob_ratio", "dist1", "dist2"]
    ].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print("\n=== By prefix ===")
display(
    fork_df.groupby("prefix")[
        ["p2", "prob_margin", "prob_ratio", "dist2"]
    ].median()
)

In [ ]:
thresholds = [
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
]

rows_threshold = []

total_forks = len(fork_df)

for thr in thresholds:
    kept = int((fork_df["p2"] >= thr).sum())

    rows_threshold.append({
        "p2_threshold": thr,
        "forks_kept": kept,
        "forks_removed": total_forks - kept,
        "kept_fraction": kept / total_forks,
    })

threshold_df = pd.DataFrame(rows_threshold)

display(threshold_df)

print("Total original fork parents:", total_forks)

In [ ]:
from collections import defaultdict

import pandas as pd
import tracksdata as td

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

SWEEP_THRESHOLDS = [
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.95,
    0.96,
    0.97,
    0.98,
]

cv_meta = pd.read_csv(
    PROJECT / "configs" / "cv_manifest.csv"
)

estimated_map = dict(
    zip(
        cv_meta["dataset"],
        cv_meta["estimated_nodes"],
    )
)

records_by_threshold = {
    thr: [] for thr in SWEEP_THRESHOLDS
}

forks_kept_by_threshold = {
    thr: 0 for thr in SWEEP_THRESHOLDS
}


def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


pred_paths = sorted(BASE_PRED_DIR.glob("*.geff"))

print("Datasets:", len(pred_paths))
assert len(pred_paths) == 40


for i, pred_path in enumerate(pred_paths, start=1):

    name = pred_path.stem

    pred_graph = load_graph(pred_path)

    gt_graph = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    n_total = float(
        estimated_map[name]
    )

    # -------------------------------------------------
    # IMPORTANT:
    # official_evaluate() performs node matching and
    # creates match_node_id, which node_recall needs.
    #
    # Do this once before modifying any edges.
    # -------------------------------------------------

    _ = official_evaluate(
        pred_graph,
        gt_graph,
        scale=scale,
        max_distance=7.0,
    )

    recall = node_recall(
        pred_graph,
        gt_graph,
    )

    # -------------------------------------------------
    # Identify second-child edges
    # -------------------------------------------------

    edge_df = pred_graph.edge_attrs(
        attr_keys=["edge_prob"],
        unpack=True,
    )

    by_source = defaultdict(list)

    for row in edge_df.iter_rows(named=True):

        source = int(row["source_id"])

        by_source[source].append({
            "edge_id": int(row["edge_id"]),
            "prob": float(row["edge_prob"]),
        })

    second_edges = []

    for source_id, edges in by_source.items():

        assert len(edges) <= 2, (
            name,
            source_id,
            len(edges),
        )

        if len(edges) == 2:

            edges = sorted(
                edges,
                key=lambda x: (
                    -x["prob"],
                    x["edge_id"],
                ),
            )

            second_edges.append({
                "edge_id": edges[1]["edge_id"],
                "p2": edges[1]["prob"],
            })

    # -------------------------------------------------
    # Progressive threshold sweep
    #
    # 0.80 -> 0.85 -> ... -> 0.98
    #
    # Higher thresholds only remove additional
    # second-child edges.
    # -------------------------------------------------

    already_removed = set()

    for thr in SWEEP_THRESHOLDS:

        remove_now = [
            e["edge_id"]
            for e in second_edges
            if (
                e["p2"] < thr
                and e["edge_id"]
                not in already_removed
            )
        ]

        if remove_now:
            pred_graph.bulk_remove_edges(
                remove_now
            )

            already_removed.update(
                remove_now
            )

        forks_kept = sum(
            e["p2"] >= thr
            for e in second_edges
        )

        forks_kept_by_threshold[thr] += (
            forks_kept
        )

        # Re-evaluate graph topology after removing edges
        er = official_evaluate(
            pred_graph,
            gt_graph,
            scale=scale,
            max_distance=7.0,
        )

        m = per_sample_metrics(
            er,
            n_total,
            recall,
        )

        records_by_threshold[thr].append(
            m
        )

    if i % 5 == 0 or i == len(pred_paths):
        print(f"Processed {i}/40")


# =====================================================
# Fold-level summaries
# =====================================================

sweep_rows = []

for thr in SWEEP_THRESHOLDS:

    s = summarise(
        records_by_threshold[thr]
    )

    sweep_rows.append({
        "p2_threshold": thr,
        "forks_kept":
            forks_kept_by_threshold[thr],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "score":
            s["score"],
    })


sweep_results = pd.DataFrame(
    sweep_rows
)

print("\n=== SELECTIVE DIVISION SWEEP ===")

display(
    sweep_results
)


best_idx = (
    sweep_results["score"]
    .idxmax()
)

best = sweep_results.loc[
    best_idx
]

print("\n=== BEST SELECTIVE THRESHOLD ===")
print(best)


OFFICIAL_SCORE = 0.6964058797724787
NO_DIV_SCORE = 0.7004404479824403

print(
    "\nGain vs official:",
    best["score"] - OFFICIAL_SCORE,
)

print(
    "Gain vs no-division:",
    best["score"] - NO_DIV_SCORE,
)

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Graph already matched, overwriting previous matching."
)

In [ ]:
from collections import defaultdict

import pandas as pd
import tracksdata as td

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

SWEEP_THRESHOLDS = [
    0.60,
    0.65,
    0.70,
    0.72,
    0.74,
    0.76,
    0.77,
    0.78,
    0.79,
    0.80,
    0.81,
    0.82,
    0.83,
    0.84,
    0.85,
]

cv_meta = pd.read_csv(
    PROJECT / "configs" / "cv_manifest.csv"
)

estimated_map = dict(
    zip(
        cv_meta["dataset"],
        cv_meta["estimated_nodes"],
    )
)

records_by_threshold = {
    thr: [] for thr in SWEEP_THRESHOLDS
}

forks_kept_by_threshold = {
    thr: 0 for thr in SWEEP_THRESHOLDS
}


def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


pred_paths = sorted(BASE_PRED_DIR.glob("*.geff"))

print("Datasets:", len(pred_paths))
assert len(pred_paths) == 40


for i, pred_path in enumerate(pred_paths, start=1):

    name = pred_path.stem

    pred_graph = load_graph(pred_path)

    gt_graph = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    n_total = float(
        estimated_map[name]
    )

    # -------------------------------------------------
    # IMPORTANT:
    # official_evaluate() performs node matching and
    # creates match_node_id, which node_recall needs.
    #
    # Do this once before modifying any edges.
    # -------------------------------------------------

    _ = official_evaluate(
        pred_graph,
        gt_graph,
        scale=scale,
        max_distance=7.0,
    )

    recall = node_recall(
        pred_graph,
        gt_graph,
    )

    # -------------------------------------------------
    # Identify second-child edges
    # -------------------------------------------------

    edge_df = pred_graph.edge_attrs(
        attr_keys=["edge_prob"],
        unpack=True,
    )

    by_source = defaultdict(list)

    for row in edge_df.iter_rows(named=True):

        source = int(row["source_id"])

        by_source[source].append({
            "edge_id": int(row["edge_id"]),
            "prob": float(row["edge_prob"]),
        })

    second_edges = []

    for source_id, edges in by_source.items():

        assert len(edges) <= 2, (
            name,
            source_id,
            len(edges),
        )

        if len(edges) == 2:

            edges = sorted(
                edges,
                key=lambda x: (
                    -x["prob"],
                    x["edge_id"],
                ),
            )

            second_edges.append({
                "edge_id": edges[1]["edge_id"],
                "p2": edges[1]["prob"],
            })

    # -------------------------------------------------
    # Progressive threshold sweep
    #
    # 0.80 -> 0.85 -> ... -> 0.98
    #
    # Higher thresholds only remove additional
    # second-child edges.
    # -------------------------------------------------

    already_removed = set()

    for thr in SWEEP_THRESHOLDS:

        remove_now = [
            e["edge_id"]
            for e in second_edges
            if (
                e["p2"] < thr
                and e["edge_id"]
                not in already_removed
            )
        ]

        if remove_now:
            pred_graph.bulk_remove_edges(
                remove_now
            )

            already_removed.update(
                remove_now
            )

        forks_kept = sum(
            e["p2"] >= thr
            for e in second_edges
        )

        forks_kept_by_threshold[thr] += (
            forks_kept
        )

        # Re-evaluate graph topology after removing edges
        er = official_evaluate(
            pred_graph,
            gt_graph,
            scale=scale,
            max_distance=7.0,
        )

        m = per_sample_metrics(
            er,
            n_total,
            recall,
        )

        records_by_threshold[thr].append(
            m
        )

    if i % 5 == 0 or i == len(pred_paths):
        print(f"Processed {i}/40")


# =====================================================
# Fold-level summaries
# =====================================================

sweep_rows = []

for thr in SWEEP_THRESHOLDS:

    s = summarise(
        records_by_threshold[thr]
    )

    sweep_rows.append({
        "p2_threshold": thr,
        "forks_kept":
            forks_kept_by_threshold[thr],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "score":
            s["score"],
    })


sweep_results = pd.DataFrame(
    sweep_rows
)

print("\n=== SELECTIVE DIVISION SWEEP ===")

display(
    sweep_results
)


best_idx = (
    sweep_results["score"]
    .idxmax()
)

best = sweep_results.loc[
    best_idx
]

print("\n=== BEST SELECTIVE THRESHOLD ===")
print(best)


OFFICIAL_SCORE = 0.6964058797724787
NO_DIV_SCORE = 0.7004404479824403

print(
    "\nGain vs official:",
    best["score"] - OFFICIAL_SCORE,
)

print(
    "Gain vs no-division:",
    best["score"] - NO_DIV_SCORE,
)

In [ ]:
sample_name = "6bba_df673a83"

pred_g = load_graph(
    BASE_PRED_DIR / f"{sample_name}.geff"
)

gt_g = load_graph(
    TRAIN_DIR / f"{sample_name}.geff"
)

scale = open_dataset(
    TRAIN_DIR / sample_name,
    load_image=False,
).scale

# Official matching
_ = official_evaluate(
    pred_g,
    gt_g,
    scale=scale,
    max_distance=7.0,
)

pred_nodes = pred_g.node_attrs(
    unpack=True
)

gt_nodes = gt_g.node_attrs(
    unpack=True
)

print("Scale:", scale)

print("\n=== Prediction node columns ===")
print(pred_nodes.columns)
display(pred_nodes.head())

print("\n=== GT node columns ===")
print(gt_nodes.columns)
display(gt_nodes.head())

print("\n=== GT division parents ===")

from collections import Counter

gt_edges = gt_g.edge_list()

out_degree = Counter(
    int(source)
    for source, target in gt_edges
)

gt_division_parents = [
    node_id
    for node_id, degree in out_degree.items()
    if degree == 2
]

print("Number of GT division parents:", len(gt_division_parents))
print("First few:", gt_division_parents[:10])

In [ ]:
import math
from collections import defaultdict

import numpy as np
import pandas as pd


def physical_vec(a, b, scale):
    """
    Vector a -> b in physical microns.
    scale order = (z, y, x)
    """
    return np.array([
        (b["z"] - a["z"]) * scale[0],
        (b["y"] - a["y"]) * scale[1],
        (b["x"] - a["x"]) * scale[2],
    ], dtype=float)


geometry_rows = []

pred_paths = sorted(BASE_PRED_DIR.glob("*.geff"))

for i, pred_path in enumerate(pred_paths, start=1):

    name = pred_path.stem
    prefix = name.split("_")[0]

    pred_g = load_graph(pred_path)
    gt_g = load_graph(TRAIN_DIR / f"{name}.geff")

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    # -------------------------------------------------
    # Official node matching
    # -------------------------------------------------
    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=scale,
        max_distance=7.0,
    )

    pred_nodes = (
        pred_g.node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    # -------------------------------------------------
    # GT topology
    # -------------------------------------------------
    gt_children = defaultdict(list)

    for source, target in gt_g.edge_list():
        gt_children[int(source)].append(int(target))

    gt_division_parents = {
        source
        for source, children in gt_children.items()
        if len(children) == 2
    }

    # -------------------------------------------------
    # Prediction edges
    # -------------------------------------------------
    edge_df = pred_g.edge_attrs(
        attr_keys=["edge_prob", "edge_dist"],
        unpack=True,
    ).to_pandas()

    for source_id, group in edge_df.groupby("source_id"):

        # We only care about predicted forks
        if len(group) != 2:
            continue

        group = group.sort_values(
            ["edge_prob", "edge_id"],
            ascending=[False, True],
        )

        e1 = group.iloc[0]
        e2 = group.iloc[1]

        source_id = int(source_id)
        target1 = int(e1["target_id"])
        target2 = int(e2["target_id"])

        parent = pred_nodes.loc[source_id]
        child1 = pred_nodes.loc[target1]
        child2 = pred_nodes.loc[target2]

        # -------------------------------------------------
        # Physical geometry
        # -------------------------------------------------
        v1 = physical_vec(parent, child1, scale)
        v2 = physical_vec(parent, child2, scale)

        d1 = float(np.linalg.norm(v1))
        d2 = float(np.linalg.norm(v2))

        daughter_sep = float(
            np.linalg.norm(v1 - v2)
        )

        if d1 > 0 and d2 > 0:
            cosine = float(
                np.dot(v1, v2) / (d1 * d2)
            )
            cosine = float(
                np.clip(cosine, -1.0, 1.0)
            )
            angle_deg = float(
                np.degrees(np.arccos(cosine))
            )
        else:
            cosine = np.nan
            angle_deg = np.nan

        distance_balance = (
            abs(d1 - d2) / max(d1 + d2, 1e-12)
        )

        # -------------------------------------------------
        # GT matching
        # -------------------------------------------------
        parent_match = int(parent["match_node_id"])
        child1_match = int(child1["match_node_id"])
        child2_match = int(child2["match_node_id"])

        parent_is_gt_division = (
            parent_match in gt_division_parents
        )

        matched_children = {
            x for x in [child1_match, child2_match]
            if x >= 0
        }

        true_children = set(
            gt_children.get(parent_match, [])
        )

        direct_true_division = (
            parent_is_gt_division
            and len(matched_children) == 2
            and matched_children == true_children
        )

        geometry_rows.append({
            "dataset": name,
            "prefix": prefix,

            "source_id": source_id,
            "target1_id": target1,
            "target2_id": target2,

            "p1": float(e1["edge_prob"]),
            "p2": float(e2["edge_prob"]),
            "prob_margin":
                float(e1["edge_prob"] - e2["edge_prob"]),
            "prob_ratio":
                float(
                    e2["edge_prob"]
                    / max(e1["edge_prob"], 1e-12)
                ),

            "d1_um": d1,
            "d2_um": d2,
            "daughter_sep_um": daughter_sep,
            "daughter_cosine": cosine,
            "daughter_angle_deg": angle_deg,
            "distance_balance": distance_balance,

            "parent_match": parent_match,
            "child1_match": child1_match,
            "child2_match": child2_match,

            "parent_is_gt_division":
                parent_is_gt_division,

            "both_children_matched":
                child1_match >= 0
                and child2_match >= 0,

            "direct_true_division":
                direct_true_division,
        })

    if i % 5 == 0 or i == len(pred_paths):
        print(f"Processed {i}/40")


fork_geometry = pd.DataFrame(
    geometry_rows
)

print("\nFork rows:", len(fork_geometry))
assert len(fork_geometry) == 44736

print(
    "Predicted forks whose parent matches a GT division:",
    fork_geometry["parent_is_gt_division"].sum(),
)

print(
    "Direct true divisions:",
    fork_geometry["direct_true_division"].sum(),
)

print("\n=== DIRECT TRUE vs OTHER FORKS ===")

cols = [
    "p2",
    "prob_margin",
    "prob_ratio",
    "d1_um",
    "d2_um",
    "daughter_sep_um",
    "daughter_cosine",
    "daughter_angle_deg",
    "distance_balance",
]

display(
    fork_geometry
    .groupby("direct_true_division")[cols]
    .median()
)

print("\n=== Direct true division rows ===")

display(
    fork_geometry[
        fork_geometry["direct_true_division"]
    ][
        [
            "dataset",
            "prefix",
            "p1",
            "p2",
            "d1_um",
            "d2_um",
            "daughter_sep_um",
            "daughter_angle_deg",
            "distance_balance",
        ]
    ]
    .sort_values("p2")
)

In [ ]:
# Geometry-rule diagnostic.
# This does NOT run the official metric yet.
# It only measures how selective several reasonable rules are.

fg = fork_geometry.copy()

fg["max_parent_daughter_um"] = fg[
    ["d1_um", "d2_um"]
].max(axis=1)

rules = {
    # Existing probability-only best
    "P80": (
        fg["p2"] >= 0.80
    ),

    # Wide probability gate + basic division geometry
    "G1": (
        (fg["p2"] >= 0.55)
        & (fg["daughter_sep_um"] >= 6.0)
        & (fg["daughter_angle_deg"] >= 100.0)
        & (fg["max_parent_daughter_um"] <= 8.0)
        & (fg["distance_balance"] <= 0.50)
    ),

    # Stronger angle
    "G2": (
        (fg["p2"] >= 0.55)
        & (fg["daughter_sep_um"] >= 6.0)
        & (fg["daughter_angle_deg"] >= 120.0)
        & (fg["max_parent_daughter_um"] <= 8.0)
        & (fg["distance_balance"] <= 0.50)
    ),

    # Stronger daughter separation
    "G3": (
        (fg["p2"] >= 0.55)
        & (fg["daughter_sep_um"] >= 8.0)
        & (fg["daughter_angle_deg"] >= 100.0)
        & (fg["max_parent_daughter_um"] <= 8.0)
        & (fg["distance_balance"] <= 0.50)
    ),

    # Moderate p2 + geometry
    "G4": (
        (fg["p2"] >= 0.60)
        & (fg["daughter_sep_um"] >= 6.0)
        & (fg["daughter_angle_deg"] >= 100.0)
        & (fg["max_parent_daughter_um"] <= 8.0)
        & (fg["distance_balance"] <= 0.50)
    ),

    # Stricter combined geometry
    "G5": (
        (fg["p2"] >= 0.60)
        & (fg["daughter_sep_um"] >= 8.0)
        & (fg["daughter_angle_deg"] >= 120.0)
        & (fg["max_parent_daughter_um"] <= 8.0)
        & (fg["distance_balance"] <= 0.50)
    ),
}


rows = []

total_direct_true = int(
    fg["direct_true_division"].sum()
)

for name, mask in rules.items():

    selected = fg[mask]

    direct_true_selected = int(
        selected["direct_true_division"].sum()
    )

    rows.append({
        "rule": name,
        "forks_kept": len(selected),
        "kept_fraction": len(selected) / len(fg),

        "direct_true_kept":
            direct_true_selected,

        "direct_true_total":
            total_direct_true,

        "direct_true_recall":
            (
                direct_true_selected
                / total_direct_true
                if total_direct_true > 0
                else 0.0
            ),

        "44b6_kept":
            int((selected["prefix"] == "44b6").sum()),

        "6bba_kept":
            int((selected["prefix"] == "6bba").sum()),
    })


geometry_rule_summary = (
    pd.DataFrame(rows)
    .sort_values("forks_kept")
    .reset_index(drop=True)
)

display(geometry_rule_summary)

In [ ]:
from collections import defaultdict

import pandas as pd
import tracksdata as td

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

# ---------------------------------------------------------
# Build selected fork-parent sets for each predefined rule
# ---------------------------------------------------------

RULE_NAMES = [
    "P80",
    "G1",
    "G2",
    "G3",
    "G4",
    "G5",
]

selected_sources = {}

for rule_name in RULE_NAMES:
    mask = rules[rule_name]

    selected_sources[rule_name] = (
        fork_geometry.loc[
            mask,
            ["dataset", "source_id"],
        ]
        .groupby("dataset")["source_id"]
        .apply(lambda s: set(map(int, s)))
        .to_dict()
    )


# ---------------------------------------------------------
# Metadata
# ---------------------------------------------------------

cv_meta = pd.read_csv(
    PROJECT / "configs" / "cv_manifest.csv"
)

estimated_map = dict(
    zip(
        cv_meta["dataset"],
        cv_meta["estimated_nodes"],
    )
)


def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


pred_paths = sorted(
    BASE_PRED_DIR.glob("*.geff")
)

assert len(pred_paths) == 40


# ---------------------------------------------------------
# Evaluate each rule independently
# ---------------------------------------------------------

records_by_rule = {
    rule_name: []
    for rule_name in RULE_NAMES
}

forks_kept_by_rule = {
    rule_name: 0
    for rule_name in RULE_NAMES
}


for rule_name in RULE_NAMES:

    print(f"\n=== Evaluating {rule_name} ===")

    for i, pred_path in enumerate(
        pred_paths,
        start=1,
    ):

        name = pred_path.stem

        pred_g = load_graph(pred_path)

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        scale = open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale

        n_total = float(
            estimated_map[name]
        )

        allowed_fork_sources = (
            selected_sources[rule_name]
            .get(name, set())
        )

        # -------------------------------------------------
        # Find the second edge for every predicted fork.
        #
        # Keep:
        #   - first/highest-prob edge always
        #   - second edge only if its source passes rule
        # -------------------------------------------------

        edge_df = pred_g.edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )

        by_source = defaultdict(list)

        for row in edge_df.iter_rows(
            named=True
        ):
            by_source[
                int(row["source_id"])
            ].append({
                "edge_id":
                    int(row["edge_id"]),

                "prob":
                    float(row["edge_prob"]),
            })

        remove_ids = []
        kept_forks_dataset = 0

        for source_id, edges in (
            by_source.items()
        ):

            assert len(edges) <= 2, (
                name,
                source_id,
                len(edges),
            )

            if len(edges) != 2:
                continue

            edges = sorted(
                edges,
                key=lambda x: (
                    -x["prob"],
                    x["edge_id"],
                ),
            )

            second_edge_id = (
                edges[1]["edge_id"]
            )

            if (
                source_id
                in allowed_fork_sources
            ):
                kept_forks_dataset += 1
            else:
                remove_ids.append(
                    second_edge_id
                )

        if remove_ids:
            pred_g.bulk_remove_edges(
                remove_ids
            )

        forks_kept_by_rule[
            rule_name
        ] += kept_forks_dataset

        # -------------------------------------------------
        # Official metric
        # -------------------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=scale,
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            n_total,
            recall,
        )

        records_by_rule[
            rule_name
        ].append(m)

        if i % 10 == 0:
            print(
                f"{rule_name}: "
                f"{i}/40"
            )


# ---------------------------------------------------------
# Fold-level summary
# ---------------------------------------------------------

official_results = []

for rule_name in RULE_NAMES:

    s = summarise(
        records_by_rule[rule_name]
    )

    official_results.append({
        "rule":
            rule_name,

        "forks_kept":
            forks_kept_by_rule[
                rule_name
            ],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "score":
            s["score"],
    })


geometry_metric_results = (
    pd.DataFrame(official_results)
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\n=== OFFICIAL GEOMETRY RULE RESULTS ==="
)

display(
    geometry_metric_results
)


print(
    "\n=== BEST GEOMETRY RULE ==="
)

best_geometry = (
    geometry_metric_results.iloc[0]
)

print(best_geometry)


OFFICIAL_BASELINE = (
    0.6964058797724787
)

NO_DIVISION = (
    0.7004404479824403
)

P80_EXPECTED = (
    0.7016076090419297
)

print(
    "\nGain vs official baseline:",
    best_geometry["score"]
    - OFFICIAL_BASELINE,
)

print(
    "Gain vs no-division:",
    best_geometry["score"]
    - NO_DIVISION,
)


# P80 should reproduce our previous result
p80_score = float(
    geometry_metric_results.loc[
        geometry_metric_results[
            "rule"
        ] == "P80",
        "score",
    ].iloc[0]
)

print(
    "\nP80 reproduction:",
    p80_score,
)

print(
    "P80 difference from prior:",
    p80_score - P80_EXPECTED,
)

assert abs(
    p80_score - P80_EXPECTED
) < 1e-8

print(
    "\nGeometry rule evaluation: OK"
)

In [ ]:
# =========================================================
# G2 / P80 domain decomposition
# =========================================================

pred_names = [
    p.stem
    for p in pred_paths
]

assert len(pred_names) == 40

cv_lookup = (
    cv_meta
    .set_index("dataset")
)

comparison_rows = []

for rule_name in ["G2", "P80"]:

    metrics = records_by_rule[
        rule_name
    ]

    assert len(metrics) == len(pred_names)

    for prefix in ["44b6", "6bba"]:

        prefix_metrics = [
            m
            for name, m in zip(
                pred_names,
                metrics,
            )
            if cv_lookup.loc[
                name,
                "prefix",
            ] == prefix
        ]

        s = summarise(
            prefix_metrics
        )

        comparison_rows.append({
            "rule": rule_name,
            "prefix": prefix,
            "n": s["n"],

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "division_tp":
                s["division_tp"],

            "division_fp":
                s["division_fp"],

            "division_fn":
                s["division_fn"],

            "node_recall":
                s["node_recall"],

            "score":
                s["score"],
        })


domain_comparison = pd.DataFrame(
    comparison_rows
)

print(
    "=== G2 vs P80 BY PREFIX ==="
)

display(
    domain_comparison
)


# Easier direct score comparison
score_pivot = (
    domain_comparison
    .pivot(
        index="prefix",
        columns="rule",
        values="score",
    )
)

score_pivot[
    "G2_minus_P80"
] = (
    score_pivot["G2"]
    - score_pivot["P80"]
)

print(
    "\n=== SCORE DELTA ==="
)

display(
    score_pivot
)

In [ ]:
# =========================================================
# Per-dataset G2 failure decomposition
# =========================================================

g2_rows = []

for name, m in zip(
    pred_names,
    records_by_rule["G2"],
):
    div_tp = m["division_tp"]
    div_fp = m["division_fp"]
    div_fn = m["division_fn"]

    div_denom = div_tp + div_fp + div_fn

    division_jaccard = (
        div_tp / div_denom
        if div_denom > 0
        else 0.0
    )

    dataset_score = (
        m["adj_edge_jaccard"]
        + 0.1 * division_jaccard
    )

    g2_rows.append({
        "dataset": name,
        "prefix": cv_lookup.loc[name, "prefix"],

        "edge_tp": m["edge_tp"],
        "edge_fp": m["edge_fp"],
        "edge_fn": m["edge_fn"],

        "edge_jaccard":
            m["edge_jaccard"],

        "adj_edge_jaccard":
            m["adj_edge_jaccard"],

        "division_tp": div_tp,
        "division_fp": div_fp,
        "division_fn": div_fn,
        "division_jaccard":
            division_jaccard,

        "node_recall":
            m["node_recall"],

        "total_node_ratio":
            m["total_node_ratio"],

        "score":
            dataset_score,
    })


g2_dataset_df = pd.DataFrame(
    g2_rows
)


print(
    "=== WORST 15 DATASETS BY G2 SCORE ==="
)

display(
    g2_dataset_df
    .sort_values("score")
    .head(15)
    .reset_index(drop=True)
)


print(
    "\n=== WORST 15 BY RAW EDGE JACCARD ==="
)

display(
    g2_dataset_df
    .sort_values("edge_jaccard")
    .head(15)
    .reset_index(drop=True)
)


print(
    "\n=== PREFIX SUMMARY ==="
)

display(
    g2_dataset_df
    .groupby("prefix")[
        [
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "total_node_ratio",
            "score",
        ]
    ]
    .agg(
        ["mean", "median", "min", "max"]
    )
)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import tracksdata as td


SCALE = np.array([1.625, 0.40625, 0.40625], dtype=float)

# MNN 最大允许距离，只用于估计 frame motion
MNN_MAX_DIST_UM = 8.0


def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


def estimate_frame_motion_mnn(graph, scale=SCALE, max_dist_um=8.0):
    """
    Estimate t -> t+1 global motion using only predicted node coordinates.

    Procedure:
    1. convert z,y,x to physical microns
    2. nearest neighbor forward and backward
    3. retain mutual NN pairs
    4. robust global shift = median displacement vector
    """

    nodes = (
        graph.node_attrs(unpack=True)
        .select(["t", "node_id", "z", "y", "x"])
        .to_pandas()
    )

    frame_rows = []

    times = sorted(nodes["t"].unique())

    for t in times[:-1]:

        a = nodes[nodes["t"] == t]
        b = nodes[nodes["t"] == t + 1]

        if len(a) == 0 or len(b) == 0:
            continue

        xyz_a = a[["z", "y", "x"]].to_numpy(float) * scale
        xyz_b = b[["z", "y", "x"]].to_numpy(float) * scale

        tree_b = cKDTree(xyz_b)
        tree_a = cKDTree(xyz_a)

        dist_ab, idx_ab = tree_b.query(xyz_a, k=1)
        dist_ba, idx_ba = tree_a.query(xyz_b, k=1)

        src_indices = np.arange(len(xyz_a))

        mutual = (
            idx_ba[idx_ab] == src_indices
        )

        valid = (
            mutual
            & (dist_ab <= max_dist_um)
        )

        if valid.sum() == 0:
            continue

        src = xyz_a[valid]
        dst = xyz_b[idx_ab[valid]]

        displacement = dst - src

        # Robust global shift
        shift = np.median(
            displacement,
            axis=0,
        )

        shift_mag = float(
            np.linalg.norm(shift)
        )

        # Residual after removing global shift
        residual = displacement - shift

        residual_mag = np.linalg.norm(
            residual,
            axis=1,
        )

        # Directional coherence:
        # how tightly MNN displacement vectors agree
        disp_norm = np.linalg.norm(
            displacement,
            axis=1,
        )

        usable = disp_norm > 1e-8

        if usable.sum() >= 2:
            unit = (
                displacement[usable]
                / disp_norm[usable, None]
            )

            coherence = float(
                np.linalg.norm(
                    unit.mean(axis=0)
                )
            )
        else:
            coherence = np.nan

        frame_rows.append({
            "t": int(t),
            "n_mnn": int(valid.sum()),

            "shift_z_um": float(shift[0]),
            "shift_y_um": float(shift[1]),
            "shift_x_um": float(shift[2]),

            "shift_mag_um": shift_mag,

            "coherence": coherence,

            "median_raw_disp_um":
                float(np.median(disp_norm)),

            "median_residual_um":
                float(np.median(residual_mag)),
        })

    return pd.DataFrame(frame_rows)


motion_rows = []

for i, pred_path in enumerate(
    sorted(BASE_PRED_DIR.glob("*.geff")),
    start=1,
):

    name = pred_path.stem

    g = load_graph(pred_path)

    frame_motion = estimate_frame_motion_mnn(
        g,
        max_dist_um=MNN_MAX_DIST_UM,
    )

    if len(frame_motion) == 0:
        continue

    motion_rows.append({
        "dataset": name,

        "motion_frames":
            len(frame_motion),

        "median_mnn_pairs":
            frame_motion["n_mnn"].median(),

        "median_shift_um":
            frame_motion["shift_mag_um"].median(),

        "p90_shift_um":
            frame_motion["shift_mag_um"].quantile(0.90),

        "p95_shift_um":
            frame_motion["shift_mag_um"].quantile(0.95),

        "max_shift_um":
            frame_motion["shift_mag_um"].max(),

        "median_coherence":
            frame_motion["coherence"].median(),

        "p90_coherence":
            frame_motion["coherence"].quantile(0.90),

        "median_residual_um":
            frame_motion["median_residual_um"].median(),
    })

    if i % 10 == 0:
        print(f"Processed {i}/40")


motion_df = pd.DataFrame(motion_rows)

association_audit = (
    g2_dataset_df
    .merge(
        motion_df,
        on="dataset",
        how="left",
        validate="one_to_one",
    )
)

print("\n=== ASSOCIATION-FOCUSED FAILURES ===")

display(
    association_audit[
        association_audit["node_recall"] >= 0.97
    ][
        [
            "dataset",
            "prefix",
            "edge_jaccard",
            "node_recall",
            "total_node_ratio",
            "median_shift_um",
            "p95_shift_um",
            "max_shift_um",
            "median_coherence",
            "median_residual_um",
        ]
    ]
    .sort_values("edge_jaccard")
    .head(15)
    .reset_index(drop=True)
)

print("\n=== CORRELATION WITH EDGE JACCARD ===")

corr_cols = [
    "edge_jaccard",
    "node_recall",
    "total_node_ratio",
    "median_shift_um",
    "p95_shift_um",
    "max_shift_um",
    "median_coherence",
    "median_residual_um",
]

display(
    association_audit[corr_cols]
    .corr(method="spearman")
    [["edge_jaccard"]]
    .sort_values("edge_jaccard")
)

关于“global coherent motion 是当前 association 主瓶颈”的假设，在实际预测上没有得到支持

In [ ]:
import pandas as pd
import tracksdata as td


isolated_rows = []

for i, pred_path in enumerate(
    sorted(BASE_PRED_DIR.glob("*.geff")),
    start=1,
):
    name = pred_path.stem

    loaded = td.graph.IndexedRXGraph.from_geff(
        pred_path
    )
    g = loaded[0] if isinstance(loaded, tuple) else loaded

    node_ids = set(
        map(
            int,
            g.node_attrs(
                unpack=True
            )["node_id"].to_list(),
        )
    )

    edge_list = g.edge_list()

    connected_ids = set()

    for source, target in edge_list:
        connected_ids.add(int(source))
        connected_ids.add(int(target))

    isolated_ids = (
        node_ids - connected_ids
    )

    n_nodes = len(node_ids)
    n_connected = len(connected_ids)
    n_isolated = len(isolated_ids)

    estimated_nodes = float(
        cv_lookup.loc[
            name,
            "estimated_nodes",
        ]
    )

    isolated_rows.append({
        "dataset": name,
        "prefix": cv_lookup.loc[name, "prefix"],

        "pred_nodes":
            n_nodes,

        "connected_nodes":
            n_connected,

        "isolated_nodes":
            n_isolated,

        "isolated_fraction":
            n_isolated / n_nodes
            if n_nodes > 0
            else 0.0,

        "estimated_nodes":
            estimated_nodes,

        "original_node_ratio":
            (
                n_nodes - estimated_nodes
            ) / estimated_nodes,

        "connected_only_ratio":
            (
                n_connected - estimated_nodes
            ) / estimated_nodes,
    })

    if i % 10 == 0:
        print(f"Processed {i}/40")


isolated_df = pd.DataFrame(
    isolated_rows
)

isolated_audit = (
    g2_dataset_df
    .merge(
        isolated_df,
        on=["dataset", "prefix"],
        how="left",
        validate="one_to_one",
    )
)


print(
    "\n=== ISOLATED NODE SUMMARY ==="
)

display(
    isolated_df.groupby("prefix")[
        [
            "pred_nodes",
            "connected_nodes",
            "isolated_nodes",
            "isolated_fraction",
            "original_node_ratio",
            "connected_only_ratio",
        ]
    ]
    .agg(
        ["mean", "median", "min", "max"]
    )
)


print(
    "\n=== WORST OVERPREDICTION DATASETS ==="
)

display(
    isolated_audit[
        [
            "dataset",
            "prefix",
            "edge_jaccard",
            "node_recall",
            "pred_nodes",
            "estimated_nodes",
            "isolated_nodes",
            "isolated_fraction",
            "original_node_ratio",
            "connected_only_ratio",
        ]
    ]
    .sort_values(
        "original_node_ratio",
        ascending=False,
    )
    .head(15)
    .reset_index(drop=True)
)


print(
    "\n=== TOTALS ==="
)

print(
    "Predicted nodes:",
    isolated_df["pred_nodes"].sum(),
)

print(
    "Connected nodes:",
    isolated_df["connected_nodes"].sum(),
)

print(
    "Isolated nodes:",
    isolated_df["isolated_nodes"].sum(),
)

print(
    "Overall isolated fraction:",
    isolated_df["isolated_nodes"].sum()
    / isolated_df["pred_nodes"].sum(),
)

In [ ]:
from collections import defaultdict

import pandas as pd
import tracksdata as td

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

# =========================================================
# G2 + post-G2 isolated-node pruning diagnostic
# =========================================================

g2_selected_sources = (
    fork_geometry.loc[
        rules["G2"],
        ["dataset", "source_id"],
    ]
    .groupby("dataset")["source_id"]
    .apply(lambda s: set(map(int, s)))
    .to_dict()
)

cv_meta = pd.read_csv(
    PROJECT / "configs" / "cv_manifest.csv"
)

estimated_map = dict(
    zip(
        cv_meta["dataset"],
        cv_meta["estimated_nodes"],
    )
)


def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


pruned_records = []
prune_stats = []

pred_paths = sorted(
    BASE_PRED_DIR.glob("*.geff")
)

assert len(pred_paths) == 40


for i, pred_path in enumerate(pred_paths, start=1):

    name = pred_path.stem

    pred_g = load_graph(pred_path)
    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    estimated_nodes = float(
        estimated_map[name]
    )

    # -----------------------------------------------------
    # 1. Apply current G2 division policy
    # -----------------------------------------------------

    allowed_fork_sources = (
        g2_selected_sources.get(name, set())
    )

    edge_df = pred_g.edge_attrs(
        attr_keys=["edge_prob"],
        unpack=True,
    )

    by_source = defaultdict(list)

    for row in edge_df.iter_rows(named=True):

        source = int(row["source_id"])

        by_source[source].append({
            "edge_id": int(row["edge_id"]),
            "prob": float(row["edge_prob"]),
        })

    remove_edges = []

    for source_id, edges in by_source.items():

        assert len(edges) <= 2

        if len(edges) != 2:
            continue

        edges = sorted(
            edges,
            key=lambda x: (
                -x["prob"],
                x["edge_id"],
            ),
        )

        # second child
        if source_id not in allowed_fork_sources:
            remove_edges.append(
                edges[1]["edge_id"]
            )

    if remove_edges:
        pred_g.bulk_remove_edges(
            remove_edges
        )

    # -----------------------------------------------------
    # 2. Find isolated nodes AFTER G2 edge pruning
    # -----------------------------------------------------

    node_ids = set(
        map(
            int,
            pred_g.node_attrs(
                unpack=True
            )["node_id"].to_list(),
        )
    )

    connected_ids = set()

    for source, target in pred_g.edge_list():
        connected_ids.add(int(source))
        connected_ids.add(int(target))

    isolated_ids = sorted(
        node_ids - connected_ids
    )

    nodes_before = pred_g.num_nodes()

    # -----------------------------------------------------
    # 3. Remove isolated nodes
    # Robust against slightly different tracksdata APIs
    # -----------------------------------------------------

    if isolated_ids:

        if hasattr(pred_g, "bulk_remove_nodes"):
            pred_g.bulk_remove_nodes(
                isolated_ids
            )

        elif hasattr(pred_g, "remove_node"):
            for node_id in isolated_ids:
                pred_g.remove_node(
                    node_id
                )

        else:
            node_remove_api = [
                x for x in dir(pred_g)
                if "remove" in x.lower()
                and "node" in x.lower()
            ]

            raise RuntimeError(
                "No known node-removal API. "
                f"Available candidates: {node_remove_api}"
            )

    nodes_after = pred_g.num_nodes()

    assert (
        nodes_before - nodes_after
        == len(isolated_ids)
    )

    # -----------------------------------------------------
    # 4. Locked official metric
    # -----------------------------------------------------

    er = official_evaluate(
        pred_g,
        gt_g,
        scale=scale,
        max_distance=7.0,
    )

    recall = node_recall(
        pred_g,
        gt_g,
    )

    m = per_sample_metrics(
        er,
        estimated_nodes,
        recall,
    )

    pruned_records.append(m)

    prune_stats.append({
        "dataset": name,
        "prefix": name.split("_")[0],

        "nodes_before":
            nodes_before,

        "isolated_removed":
            len(isolated_ids),

        "nodes_after":
            nodes_after,

        "removed_fraction":
            (
                len(isolated_ids)
                / nodes_before
                if nodes_before > 0
                else 0.0
            ),

        "node_recall_after":
            recall,

        "node_ratio_after":
            m["total_node_ratio"],

        "edge_jaccard":
            m["edge_jaccard"],

        "adj_edge_jaccard":
            m["adj_edge_jaccard"],
    })

    if i % 10 == 0:
        print(f"Processed {i}/40")


# =========================================================
# Fold-level result
# =========================================================

pruned_summary = summarise(
    pruned_records
)

prune_stats_df = pd.DataFrame(
    prune_stats
)

print(
    "\n=== G2 + ISOLATED PRUNING ==="
)

for k, v in pruned_summary.items():
    print(f"{k}: {v}")


print(
    "\n=== NODE CLEANUP ==="
)

print(
    "Nodes removed:",
    prune_stats_df[
        "isolated_removed"
    ].sum(),
)

print(
    "Median removed fraction:",
    prune_stats_df[
        "removed_fraction"
    ].median(),
)

print(
    "Median node ratio after:",
    prune_stats_df[
        "node_ratio_after"
    ].median(),
)


print(
    "\n=== RECALL CHANGE ==="
)

print(
    "Mean node recall after:",
    prune_stats_df[
        "node_recall_after"
    ].mean(),
)

print(
    "Min node recall after:",
    prune_stats_df[
        "node_recall_after"
    ].min(),
)


G2_SCORE = 0.7029083146320184

print(
    "\nScore delta vs G2:",
    pruned_summary["score"]
    - G2_SCORE,
)

diagnostic best 已经从： 0.702908 → 0.737173

In [ ]:
from collections import defaultdict
import pandas as pd
import tracksdata as td

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

VARIANTS = [
    "g2_only",
    "original_isolated",
    "g2_created_isolated",
    "all_isolated",
]

records_by_cleanup = {
    v: [] for v in VARIANTS
}

removed_by_cleanup = {
    v: 0 for v in VARIANTS
}


def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


def isolated_node_ids(graph):
    node_ids = set(
        map(
            int,
            graph.node_attrs(unpack=True)["node_id"].to_list(),
        )
    )

    connected = set()

    for s, t in graph.edge_list():
        connected.add(int(s))
        connected.add(int(t))

    return node_ids - connected


def apply_g2(graph, dataset):
    allowed = g2_selected_sources.get(
        dataset,
        set(),
    )

    edge_df = graph.edge_attrs(
        attr_keys=["edge_prob"],
        unpack=True,
    )

    by_source = defaultdict(list)

    for row in edge_df.iter_rows(named=True):
        by_source[int(row["source_id"])].append({
            "edge_id": int(row["edge_id"]),
            "prob": float(row["edge_prob"]),
        })

    remove_edges = []

    for source_id, edges in by_source.items():

        assert len(edges) <= 2

        if len(edges) != 2:
            continue

        edges = sorted(
            edges,
            key=lambda x: (
                -x["prob"],
                x["edge_id"],
            ),
        )

        if source_id not in allowed:
            remove_edges.append(
                edges[1]["edge_id"]
            )

    if remove_edges:
        graph.bulk_remove_edges(
            remove_edges
        )


pred_paths = sorted(
    BASE_PRED_DIR.glob("*.geff")
)

for i, pred_path in enumerate(pred_paths, start=1):

    name = pred_path.stem

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    estimated_nodes = float(
        estimated_map[name]
    )

    # --------------------------------------------------
    # Determine the two isolated-node populations once
    # --------------------------------------------------

    original_graph = load_graph(
        pred_path
    )

    original_iso = isolated_node_ids(
        original_graph
    )

    apply_g2(
        original_graph,
        name,
    )

    after_g2_iso = isolated_node_ids(
        original_graph
    )

    g2_created_iso = (
        after_g2_iso - original_iso
    )

    assert original_iso <= after_g2_iso

    # --------------------------------------------------
    # Evaluate each cleanup independently from fresh graph
    # --------------------------------------------------

    removal_sets = {
        "g2_only":
            set(),

        "original_isolated":
            original_iso,

        "g2_created_isolated":
            g2_created_iso,

        "all_isolated":
            after_g2_iso,
    }

    for variant in VARIANTS:

        pred_g = load_graph(
            pred_path
        )

        apply_g2(
            pred_g,
            name,
        )

        remove_nodes = sorted(
            removal_sets[variant]
        )

        if remove_nodes:
            pred_g.bulk_remove_nodes(
                remove_nodes
            )

        removed_by_cleanup[
            variant
        ] += len(remove_nodes)

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=scale,
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            estimated_nodes,
            recall,
        )

        records_by_cleanup[
            variant
        ].append(m)

    if i % 10 == 0:
        print(f"Processed {i}/40")


# ======================================================
# Fold summaries
# ======================================================

cleanup_rows = []

for variant in VARIANTS:

    s = summarise(
        records_by_cleanup[
            variant
        ]
    )

    cleanup_rows.append({
        "variant":
            variant,

        "nodes_removed":
            removed_by_cleanup[
                variant
            ],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "score":
            s["score"],
    })


cleanup_ablation = (
    pd.DataFrame(cleanup_rows)
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\n=== ISOLATED-NODE SOURCE ABLATION ==="
)

display(
    cleanup_ablation
)

print("\nOriginal isolated nodes:", sum(
    len(isolated_node_ids(load_graph(p)))
    for p in pred_paths
))

print(
    "Additional nodes becoming isolated after G2:",
    removed_by_cleanup["g2_created_isolated"],
)

G2-created isolated nodes 几乎可以视为低风险垃圾节点

In [ ]:
import numpy as np
import pandas as pd
import tracksdata as td

from tracking_cellmot.io import open_dataset


isolated_match_rows = []

for i, pred_path in enumerate(
    sorted(BASE_PRED_DIR.glob("*.geff")),
    start=1,
):
    name = pred_path.stem

    # --------------------------------------------------
    # Baseline graph: identify original isolated nodes
    # --------------------------------------------------
    base_g = load_graph(pred_path)

    original_iso = isolated_node_ids(base_g)

    # --------------------------------------------------
    # Apply G2: identify newly-created isolated nodes
    # --------------------------------------------------
    g2_g = load_graph(pred_path)

    apply_g2(g2_g, name)

    post_g2_iso = isolated_node_ids(g2_g)

    created_iso = (
        post_g2_iso - original_iso
    )

    assert original_iso <= post_g2_iso

    # --------------------------------------------------
    # Official matching on the G2 graph
    # before deleting any nodes
    # --------------------------------------------------
    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    _ = official_evaluate(
        g2_g,
        gt_g,
        scale=scale,
        max_distance=7.0,
    )

    nodes = (
        g2_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    max_t = int(nodes["t"].max())

    # --------------------------------------------------
    # Record every post-G2 isolated node
    # --------------------------------------------------
    for node_id in post_g2_iso:

        row = nodes.loc[node_id]

        t = int(row["t"])
        match_id = int(row["match_node_id"])
        match_score = float(row["match_score"])

        if node_id in original_iso:
            source_type = "original_isolated"
        else:
            source_type = "g2_created_isolated"

        distance_to_boundary = min(
            t,
            max_t - t,
        )

        isolated_match_rows.append({
            "dataset": name,
            "prefix": name.split("_")[0],

            "node_id": int(node_id),
            "source_type": source_type,

            "t": t,
            "max_t": max_t,
            "distance_to_boundary":
                distance_to_boundary,

            "matched_gt":
                match_id >= 0,

            "match_node_id":
                match_id,

            "match_score":
                match_score,
        })

    if i % 10 == 0:
        print(f"Processed {i}/40")


isolated_match_df = pd.DataFrame(
    isolated_match_rows
)

print(
    "\n=== MATCH RATE BY ISOLATED SOURCE ==="
)

source_summary = (
    isolated_match_df
    .groupby("source_type")
    .agg(
        nodes=("node_id", "size"),
        matched=("matched_gt", "sum"),
        match_rate=("matched_gt", "mean"),
        median_match_score=("match_score", "median"),
    )
)

display(source_summary)


# ------------------------------------------------------
# Temporal buckets
# ------------------------------------------------------

def boundary_bucket(d):
    if d == 0:
        return "boundary_0"
    elif d <= 1:
        return "within_1"
    elif d <= 2:
        return "within_2"
    elif d <= 5:
        return "within_5"
    elif d <= 10:
        return "within_10"
    else:
        return "interior"


isolated_match_df["boundary_bucket"] = (
    isolated_match_df[
        "distance_to_boundary"
    ].map(boundary_bucket)
)


print(
    "\n=== MATCH RATE BY TIME POSITION ==="
)

time_summary = (
    isolated_match_df
    .groupby(
        ["source_type", "boundary_bucket"],
        observed=True,
    )
    .agg(
        nodes=("node_id", "size"),
        matched=("matched_gt", "sum"),
        match_rate=("matched_gt", "mean"),
    )
    .reset_index()
)

display(time_summary)


print(
    "\n=== ORIGINAL ISOLATED: MATCHED vs UNMATCHED ==="
)

display(
    isolated_match_df[
        isolated_match_df["source_type"]
        == "original_isolated"
    ]
    .groupby("matched_gt")[
        [
            "t",
            "distance_to_boundary",
            "match_score",
        ]
    ]
    .agg(
        ["mean", "median", "min", "max"]
    )
)

original isolated 的 GT match rate 有 0.99

G2-created isolated 的 GT match rate 有 1.33%

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd

MIN_COMPONENT_SIZES = [1, 2, 3, 4, 5]

records_by_min_size = {
    k: [] for k in MIN_COMPONENT_SIZES
}

nodes_removed_by_min_size = {
    k: 0 for k in MIN_COMPONENT_SIZES
}


def component_sets(graph):
    """
    Weakly connected components using union-find.
    Works directly from node IDs + edge list.
    """

    node_ids = list(
        map(
            int,
            graph.node_attrs(
                unpack=True
            )["node_id"].to_list(),
        )
    )

    parent = {
        n: n for n in node_ids
    }

    size = {
        n: 1 for n in node_ids
    }

    def find(x):
        root = x

        while parent[root] != root:
            root = parent[root]

        while parent[x] != x:
            nxt = parent[x]
            parent[x] = root
            x = nxt

        return root

    def union(a, b):
        ra = find(a)
        rb = find(b)

        if ra == rb:
            return

        if size[ra] < size[rb]:
            ra, rb = rb, ra

        parent[rb] = ra
        size[ra] += size[rb]

    for source, target in graph.edge_list():
        union(
            int(source),
            int(target),
        )

    components = defaultdict(list)

    for node_id in node_ids:
        components[
            find(node_id)
        ].append(node_id)

    return list(
        components.values()
    )


pred_paths = sorted(
    BASE_PRED_DIR.glob("*.geff")
)

assert len(pred_paths) == 40


for i, pred_path in enumerate(
    pred_paths,
    start=1,
):
    name = pred_path.stem

    # --------------------------------------------------
    # Build G2 graph once to determine components
    # --------------------------------------------------

    g2_graph = load_graph(
        pred_path
    )

    apply_g2(
        g2_graph,
        name,
    )

    components = component_sets(
        g2_graph
    )

    component_sizes = np.array(
        [len(c) for c in components]
    )

    # Precompute node-removal sets
    removal_sets = {}

    for min_size in MIN_COMPONENT_SIZES:

        remove_nodes = []

        for comp in components:
            if len(comp) < min_size:
                remove_nodes.extend(
                    comp
                )

        removal_sets[
            min_size
        ] = remove_nodes

    # --------------------------------------------------
    # Evaluate every threshold from a fresh graph
    # --------------------------------------------------

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    estimated_nodes = float(
        estimated_map[name]
    )

    for min_size in MIN_COMPONENT_SIZES:

        pred_g = load_graph(
            pred_path
        )

        apply_g2(
            pred_g,
            name,
        )

        remove_nodes = (
            removal_sets[min_size]
        )

        if remove_nodes:
            pred_g.bulk_remove_nodes(
                remove_nodes
            )

        nodes_removed_by_min_size[
            min_size
        ] += len(remove_nodes)

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=scale,
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            estimated_nodes,
            recall,
        )

        records_by_min_size[
            min_size
        ].append(m)

    if i % 10 == 0:
        print(f"Processed {i}/40")


# ======================================================
# Fold summary
# ======================================================

component_rows = []

for min_size in MIN_COMPONENT_SIZES:

    records = records_by_min_size[
        min_size
    ]

    s = summarise(records)

    component_rows.append({
        "min_component_nodes":
            min_size,

        "nodes_removed":
            nodes_removed_by_min_size[
                min_size
            ],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


component_pruning_results = (
    pd.DataFrame(component_rows)
    .sort_values(
        "min_component_nodes"
    )
    .reset_index(drop=True)
)

print(
    "\n=== SHORT-COMPONENT PRUNING SWEEP ==="
)

display(
    component_pruning_results
)


# Validation:
# min size = 2 should reproduce G2 + all isolated pruning.

SIZE2_EXPECTED = 0.7371729284318596

size2_score = float(
    component_pruning_results.loc[
        component_pruning_results[
            "min_component_nodes"
        ] == 2,
        "score",
    ].iloc[0]
)

print(
    "\nSize-2 reproduction:",
    size2_score,
)

print(
    "Difference:",
    size2_score
    - SIZE2_EXPECTED,
)

assert abs(
    size2_score - SIZE2_EXPECTED
) < 1e-8

print(
    "\nComponent pruning sweep: OK"
)

G2 division geometry
+
remove singleton components only

In [ ]:
import numpy as np
import pandas as pd

two_node_rows = []

for i, pred_path in enumerate(
    sorted(BASE_PRED_DIR.glob("*.geff")),
    start=1,
):
    name = pred_path.stem

    pred_g = load_graph(pred_path)
    apply_g2(pred_g, name)

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    # Official matching
    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=scale,
        max_distance=7.0,
    )

    nodes_df = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges_df = (
        pred_g
        .edge_attrs(
            attr_keys=["edge_prob", "edge_dist"],
            unpack=True,
        )
        .to_pandas()
    )

    # Edge lookup by unordered node pair
    edge_lookup = {}

    for _, e in edges_df.iterrows():
        s = int(e["source_id"])
        t = int(e["target_id"])

        edge_lookup[frozenset([s, t])] = e

    components = component_sets(
        pred_g
    )

    for comp in components:

        if len(comp) != 2:
            continue

        n1, n2 = map(int, comp)

        key = frozenset([n1, n2])

        # A connected 2-node component should have exactly one edge
        if key not in edge_lookup:
            continue

        e = edge_lookup[key]

        source = int(e["source_id"])
        target = int(e["target_id"])

        source_row = nodes_df.loc[source]
        target_row = nodes_df.loc[target]

        source_match = int(
            source_row["match_node_id"]
        )

        target_match = int(
            target_row["match_node_id"]
        )

        # Conservative diagnostic:
        # does matched GT topology contain this exact edge?
        gt_edge_set = {
            (int(s), int(t))
            for s, t in gt_g.edge_list()
        }

        direct_true_edge = (
            source_match >= 0
            and target_match >= 0
            and (
                source_match,
                target_match,
            ) in gt_edge_set
        )

        two_node_rows.append({
            "dataset": name,
            "prefix": name.split("_")[0],

            "source_id": source,
            "target_id": target,

            "edge_prob":
                float(e["edge_prob"]),

            "edge_dist":
                float(e["edge_dist"]),

            "source_matched":
                source_match >= 0,

            "target_matched":
                target_match >= 0,

            "both_nodes_matched":
                (
                    source_match >= 0
                    and target_match >= 0
                ),

            "direct_true_edge":
                direct_true_edge,
        })

    if i % 10 == 0:
        print(f"Processed {i}/40")


two_node_df = pd.DataFrame(
    two_node_rows
)

print("\n2-node components:", len(two_node_df))

print(
    "Direct true GT edges:",
    two_node_df[
        "direct_true_edge"
    ].sum(),
)

print("\n=== TRUE vs OTHER 2-NODE TRACKLETS ===")

display(
    two_node_df
    .groupby("direct_true_edge")[
        [
            "edge_prob",
            "edge_dist",
            "both_nodes_matched",
        ]
    ]
    .agg(
        ["count", "mean", "median", "min", "max"]
    )
)

print("\n=== EDGE PROB QUANTILES ===")

display(
    two_node_df
    .groupby("direct_true_edge")[
        "edge_prob"
    ]
    .quantile(
        [0.1, 0.25, 0.5, 0.75, 0.9]
    )
    .unstack()
)

edge_prob 对 2-node tracklet 的真假区分能力很弱，不能靠 probability threshold 解决

In [ ]:
# =========================================================
# Physical displacement audit for 2-node components
# =========================================================

physical_rows = []

for i, pred_path in enumerate(
    sorted(BASE_PRED_DIR.glob("*.geff")),
    start=1,
):
    name = pred_path.stem

    pred_g = load_graph(pred_path)
    apply_g2(pred_g, name)

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = np.array(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    # Official node matching for diagnostic label
    _ = official_evaluate(
        pred_g,
        gt_g,
        scale=tuple(scale),
        max_distance=7.0,
    )

    nodes = (
        pred_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        pred_g
        .edge_attrs(
            attr_keys=["edge_prob", "edge_dist"],
            unpack=True,
        )
        .to_pandas()
    )

    gt_edge_set = {
        (int(s), int(t))
        for s, t in gt_g.edge_list()
    }

    # lookup source/target pair -> edge row
    edge_lookup = {
        frozenset([
            int(e["source_id"]),
            int(e["target_id"]),
        ]): e
        for _, e in edges.iterrows()
    }

    components = component_sets(pred_g)

    for comp in components:

        if len(comp) != 2:
            continue

        n1, n2 = map(int, comp)

        key = frozenset([n1, n2])

        if key not in edge_lookup:
            continue

        e = edge_lookup[key]

        source = int(e["source_id"])
        target = int(e["target_id"])

        a = nodes.loc[source]
        b = nodes.loc[target]

        delta_vox = np.array([
            float(b["z"] - a["z"]),
            float(b["y"] - a["y"]),
            float(b["x"] - a["x"]),
        ])

        delta_um = delta_vox * scale

        physical_dist_um = float(
            np.linalg.norm(delta_um)
        )

        source_match = int(
            a["match_node_id"]
        )

        target_match = int(
            b["match_node_id"]
        )

        direct_true_edge = (
            source_match >= 0
            and target_match >= 0
            and (
                source_match,
                target_match,
            ) in gt_edge_set
        )

        physical_rows.append({
            "dataset": name,
            "prefix": name.split("_")[0],

            "edge_prob":
                float(e["edge_prob"]),

            "edge_dist_raw":
                float(e["edge_dist"]),

            "physical_dist_um":
                physical_dist_um,

            "delta_t":
                int(b["t"] - a["t"]),

            "direct_true_edge":
                direct_true_edge,
        })

    if i % 10 == 0:
        print(f"Processed {i}/40")


two_node_physical = pd.DataFrame(
    physical_rows
)

assert len(two_node_physical) == 49128


print("\n=== PHYSICAL DISTANCE QUANTILES ===")

display(
    two_node_physical
    .groupby("direct_true_edge")[
        "physical_dist_um"
    ]
    .quantile(
        [
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .unstack()
)


print("\n=== DISTANCE THRESHOLD DIAGNOSTIC ===")

distance_thresholds = [
    3.0,
    4.0,
    5.0,
    6.0,
    7.0,
    8.0,
    10.0,
]

diag_rows = []

total_true = int(
    two_node_physical[
        "direct_true_edge"
    ].sum()
)

total_false = (
    len(two_node_physical)
    - total_true
)

for thr in distance_thresholds:

    remove = (
        two_node_physical[
            "physical_dist_um"
        ] > thr
    )

    true_removed = int(
        (
            remove
            & two_node_physical[
                "direct_true_edge"
            ]
        ).sum()
    )

    false_removed = int(
        (
            remove
            & ~two_node_physical[
                "direct_true_edge"
            ]
        ).sum()
    )

    diag_rows.append({
        "threshold_um": thr,

        "tracklets_removed":
            int(remove.sum()),

        "false_removed":
            false_removed,

        "true_removed":
            true_removed,

        "false_removal_rate":
            false_removed / total_false,

        "true_removal_rate":
            true_removed / total_true,
    })


distance_diag = pd.DataFrame(
    diag_rows
)

display(distance_diag)


print("\n=== DELTA-T CHECK ===")

display(
    two_node_physical[
        "delta_t"
    ].value_counts()
    .sort_index()
)

2-node tracklet 的“长距离尾部”很可能是高纯度噪声

In [ ]:
import numpy as np
import pandas as pd

TWO_NODE_DISTANCE_THRESHOLDS = [
    5.0,
    6.0,
    7.0,
    8.0,
    10.0,
    np.inf,   # singleton-only reference
]

records_by_dist = {
    thr: [] for thr in TWO_NODE_DISTANCE_THRESHOLDS
}

removed_nodes_by_dist = {
    thr: 0 for thr in TWO_NODE_DISTANCE_THRESHOLDS
}

removed_tracklets_by_dist = {
    thr: 0 for thr in TWO_NODE_DISTANCE_THRESHOLDS
}


pred_paths = sorted(
    BASE_PRED_DIR.glob("*.geff")
)

assert len(pred_paths) == 40


for i, pred_path in enumerate(
    pred_paths,
    start=1,
):
    name = pred_path.stem

    # ==================================================
    # Build G2 graph once to identify components
    # ==================================================

    analysis_g = load_graph(
        pred_path
    )

    apply_g2(
        analysis_g,
        name,
    )

    scale = np.array(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    nodes_df = (
        analysis_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges_df = (
        analysis_g
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    # unordered node-pair -> edge
    edge_lookup = {}

    for _, e in edges_df.iterrows():
        s = int(e["source_id"])
        t = int(e["target_id"])

        edge_lookup[
            frozenset([s, t])
        ] = e

    components = component_sets(
        analysis_g
    )

    singleton_nodes = []

    two_node_components = []

    for comp in components:

        comp = list(
            map(int, comp)
        )

        if len(comp) == 1:

            singleton_nodes.extend(
                comp
            )

        elif len(comp) == 2:

            n1, n2 = comp

            key = frozenset(
                [n1, n2]
            )

            if key not in edge_lookup:
                continue

            e = edge_lookup[key]

            source = int(
                e["source_id"]
            )

            target = int(
                e["target_id"]
            )

            a = nodes_df.loc[source]
            b = nodes_df.loc[target]

            delta_um = (
                np.array([
                    float(
                        b["z"] - a["z"]
                    ),
                    float(
                        b["y"] - a["y"]
                    ),
                    float(
                        b["x"] - a["x"]
                    ),
                ])
                * scale
            )

            distance_um = float(
                np.linalg.norm(
                    delta_um
                )
            )

            two_node_components.append({
                "nodes": [n1, n2],
                "distance_um":
                    distance_um,
            })

    # ==================================================
    # Evaluate thresholds independently
    # ==================================================

    estimated_nodes = float(
        estimated_map[name]
    )

    for thr in TWO_NODE_DISTANCE_THRESHOLDS:

        pred_g = load_graph(
            pred_path
        )

        apply_g2(
            pred_g,
            name,
        )

        remove_nodes = set(
            singleton_nodes
        )

        removed_tracklets = 0

        if np.isfinite(thr):

            for item in two_node_components:

                if (
                    item["distance_um"]
                    > thr
                ):
                    remove_nodes.update(
                        item["nodes"]
                    )

                    removed_tracklets += 1

        remove_nodes = sorted(
            remove_nodes
        )

        if remove_nodes:
            pred_g.bulk_remove_nodes(
                remove_nodes
            )

        removed_nodes_by_dist[
            thr
        ] += len(remove_nodes)

        removed_tracklets_by_dist[
            thr
        ] += removed_tracklets

        # Fresh GT for clean evaluation
        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            estimated_nodes,
            recall,
        )

        records_by_dist[
            thr
        ].append(m)

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


# ======================================================
# Fold summary
# ======================================================

distance_rows = []

for thr in TWO_NODE_DISTANCE_THRESHOLDS:

    records = records_by_dist[
        thr
    ]

    s = summarise(
        records
    )

    distance_rows.append({
        "max_2node_distance_um":
            thr,

        "tracklets_removed":
            removed_tracklets_by_dist[
                thr
            ],

        "nodes_removed":
            removed_nodes_by_dist[
                thr
            ],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


two_node_distance_results = (
    pd.DataFrame(
        distance_rows
    )
    .sort_values(
        "max_2node_distance_um"
    )
    .reset_index(drop=True)
)

print(
    "\n=== SINGLETON + 2-NODE DISTANCE PRUNING ==="
)

display(
    two_node_distance_results
)


# singleton-only must reproduce current best
SINGLETON_EXPECTED = (
    0.7371729284318596
)

singleton_score = float(
    two_node_distance_results.loc[
        np.isinf(
            two_node_distance_results[
                "max_2node_distance_um"
            ]
        ),
        "score",
    ].iloc[0]
)

print(
    "\nSingleton-only reproduction:",
    singleton_score,
)

print(
    "Difference:",
    singleton_score
    - SINGLETON_EXPECTED,
)

assert abs(
    singleton_score
    - SINGLETON_EXPECTED
) < 1e-8

print(
    "\n2-node distance pruning sweep: OK"
)

G2 + singleton pruning + remove 2-node tracklet if distance>5μm​

In [ ]:
import numpy as np
import pandas as pd

TWO_NODE_DISTANCE_THRESHOLDS = [
    3.0,
    3.5,
    4.0,
    4.5,
    5.0,
    5.5,
    6.0,
    np.inf,
]

records_by_dist = {
    thr: [] for thr in TWO_NODE_DISTANCE_THRESHOLDS
}

removed_nodes_by_dist = {
    thr: 0 for thr in TWO_NODE_DISTANCE_THRESHOLDS
}

removed_tracklets_by_dist = {
    thr: 0 for thr in TWO_NODE_DISTANCE_THRESHOLDS
}


pred_paths = sorted(
    BASE_PRED_DIR.glob("*.geff")
)

assert len(pred_paths) == 40


for i, pred_path in enumerate(
    pred_paths,
    start=1,
):
    name = pred_path.stem

    # ==================================================
    # Build G2 graph once to identify components
    # ==================================================

    analysis_g = load_graph(
        pred_path
    )

    apply_g2(
        analysis_g,
        name,
    )

    scale = np.array(
        open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale,
        dtype=float,
    )

    nodes_df = (
        analysis_g
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges_df = (
        analysis_g
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    # unordered node-pair -> edge
    edge_lookup = {}

    for _, e in edges_df.iterrows():
        s = int(e["source_id"])
        t = int(e["target_id"])

        edge_lookup[
            frozenset([s, t])
        ] = e

    components = component_sets(
        analysis_g
    )

    singleton_nodes = []

    two_node_components = []

    for comp in components:

        comp = list(
            map(int, comp)
        )

        if len(comp) == 1:

            singleton_nodes.extend(
                comp
            )

        elif len(comp) == 2:

            n1, n2 = comp

            key = frozenset(
                [n1, n2]
            )

            if key not in edge_lookup:
                continue

            e = edge_lookup[key]

            source = int(
                e["source_id"]
            )

            target = int(
                e["target_id"]
            )

            a = nodes_df.loc[source]
            b = nodes_df.loc[target]

            delta_um = (
                np.array([
                    float(
                        b["z"] - a["z"]
                    ),
                    float(
                        b["y"] - a["y"]
                    ),
                    float(
                        b["x"] - a["x"]
                    ),
                ])
                * scale
            )

            distance_um = float(
                np.linalg.norm(
                    delta_um
                )
            )

            two_node_components.append({
                "nodes": [n1, n2],
                "distance_um":
                    distance_um,
            })

    # ==================================================
    # Evaluate thresholds independently
    # ==================================================

    estimated_nodes = float(
        estimated_map[name]
    )

    for thr in TWO_NODE_DISTANCE_THRESHOLDS:

        pred_g = load_graph(
            pred_path
        )

        apply_g2(
            pred_g,
            name,
        )

        remove_nodes = set(
            singleton_nodes
        )

        removed_tracklets = 0

        if np.isfinite(thr):

            for item in two_node_components:

                if (
                    item["distance_um"]
                    > thr
                ):
                    remove_nodes.update(
                        item["nodes"]
                    )

                    removed_tracklets += 1

        remove_nodes = sorted(
            remove_nodes
        )

        if remove_nodes:
            pred_g.bulk_remove_nodes(
                remove_nodes
            )

        removed_nodes_by_dist[
            thr
        ] += len(remove_nodes)

        removed_tracklets_by_dist[
            thr
        ] += removed_tracklets

        # Fresh GT for clean evaluation
        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            estimated_nodes,
            recall,
        )

        records_by_dist[
            thr
        ].append(m)

    if i % 10 == 0:
        print(
            f"Processed {i}/40"
        )


# ======================================================
# Fold summary
# ======================================================

distance_rows = []

for thr in TWO_NODE_DISTANCE_THRESHOLDS:

    records = records_by_dist[
        thr
    ]

    s = summarise(
        records
    )

    distance_rows.append({
        "max_2node_distance_um":
            thr,

        "tracklets_removed":
            removed_tracklets_by_dist[
                thr
            ],

        "nodes_removed":
            removed_nodes_by_dist[
                thr
            ],

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


two_node_distance_results = (
    pd.DataFrame(
        distance_rows
    )
    .sort_values(
        "max_2node_distance_um"
    )
    .reset_index(drop=True)
)

print(
    "\n=== SINGLETON + 2-NODE DISTANCE PRUNING ==="
)

display(
    two_node_distance_results
)


# singleton-only must reproduce current best
SINGLETON_EXPECTED = (
    0.7371729284318596
)

singleton_score = float(
    two_node_distance_results.loc[
        np.isinf(
            two_node_distance_results[
                "max_2node_distance_um"
            ]
        ),
        "score",
    ].iloc[0]
)

print(
    "\nSingleton-only reproduction:",
    singleton_score,
)

print(
    "Difference:",
    singleton_score
    - SINGLETON_EXPECTED,
)

assert abs(
    singleton_score
    - SINGLETON_EXPECTED
) < 1e-8

print(
    "\n2-node distance pruning sweep: OK"
)

严格按 Fold 0 数值，最高是：0.738780@ 3.0μm

In [ ]:
from pathlib import Path
import os
import sys
import time
import subprocess

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
    / "biohub-cell-tracking-during-development/train"
)

SPLITS = PROJECT / "configs/dataset_splits.json"

CKPT = (
    PROJECT
    / "models/official_baseline/"
    / "official_baseline_fold0_best.pth"
)

PREDICT_SCRIPT = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py"
)

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    str(PROJECT / "scripts"),
    str(PROJECT / "external/official/scripts"),
    str(PROJECT / "external/official/src"),
    str(PROJECT / "src"),
])

env["USER"] = "markdd520"
env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


RUNS = [
    (0.9925, "official_baseline_fold0_det09925"),
    (0.9950, "official_baseline_fold0_det09950"),
]


for threshold, method in RUNS:

    print("\n" + "=" * 70)
    print(
        f"Running Fold 0 | "
        f"det_threshold={threshold} | "
        f"method={method}"
    )
    print("=" * 70)

    cmd = [
        sys.executable,
        str(PREDICT_SCRIPT),

        "--data-dir",
        str(TRAIN_DIR),

        "--splits",
        str(SPLITS),

        "--split",
        "0",

        "--method",
        method,

        "--weights",
        str(CKPT),

        "--det-threshold",
        str(threshold),
    ]

    start = time.time()

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()

    wall = time.time() - start

    print(
        f"\nReturn code: {return_code}"
    )

    print(
        f"Wall time: "
        f"{wall:.1f}s = {wall / 60:.2f} min"
    )

    assert return_code == 0

    pred_dir = (
        PROJECT
        / "predictions"
        / "markdd520"
        / method
        / "split_0"
    )

    pred_files = sorted(
        pred_dir.glob("*.geff")
    )

    print(
        "Predicted GEFFs:",
        len(pred_files),
    )

    assert len(pred_files) == 40


print(
    "\nDetection-threshold inference sweep: OK"
)

In [ ]:
from pathlib import Path
from collections import defaultdict
import warnings

import numpy as np
import pandas as pd
import tracksdata as td

warnings.filterwarnings(
    "ignore",
    message="Graph already matched, overwriting previous matching."
)

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
    / "biohub-cell-tracking-during-development/train"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

OFFICIAL_SRC = (
    PROJECT
    / "external/official/src"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

import sys

for p in [OFFICIAL_SRC, OFFICIAL_SCRIPTS]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)


# =========================================================
# Prediction variants
# =========================================================

PREDICTION_VARIANTS = {
    0.9900: (
        PROJECT
        / "predictions/markdd520/"
        / "official_baseline_fold0/split_0"
    ),

    0.9925: (
        PROJECT
        / "predictions/markdd520/"
        / "official_baseline_fold0_det09925/split_0"
    ),

    0.9950: (
        PROJECT
        / "predictions/markdd520/"
        / "official_baseline_fold0_det09950/split_0"
    ),
}


# =========================================================
# Frozen policy
# =========================================================

G2_P2_MIN = 0.55
G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

TWO_NODE_MAX_DIST_UM = 3.5


# =========================================================
# Metadata
# =========================================================

cv_meta = pd.read_csv(CV_PATH)

fold0 = cv_meta[
    cv_meta["fold"] == 0
].copy()

estimated_map = dict(
    zip(
        fold0["dataset"],
        fold0["estimated_nodes"],
    )
)

prefix_map = dict(
    zip(
        fold0["dataset"],
        fold0["prefix"],
    )
)

assert len(fold0) == 40


# =========================================================
# Helpers
# =========================================================

def load_graph(path):
    loaded = td.graph.IndexedRXGraph.from_geff(path)
    return loaded[0] if isinstance(loaded, tuple) else loaded


def component_sets(graph):
    node_ids = list(
        map(
            int,
            graph.node_attrs(
                unpack=True
            )["node_id"].to_list(),
        )
    )

    parent = {
        n: n for n in node_ids
    }

    size = {
        n: 1 for n in node_ids
    }

    def find(x):
        root = x

        while parent[root] != root:
            root = parent[root]

        while parent[x] != x:
            nxt = parent[x]
            parent[x] = root
            x = nxt

        return root

    def union(a, b):
        ra = find(a)
        rb = find(b)

        if ra == rb:
            return

        if size[ra] < size[rb]:
            ra, rb = rb, ra

        parent[rb] = ra
        size[ra] += size[rb]

    for source, target in graph.edge_list():
        union(
            int(source),
            int(target),
        )

    comps = defaultdict(list)

    for node_id in node_ids:
        comps[
            find(node_id)
        ].append(node_id)

    return list(comps.values())


def apply_frozen_g2(graph, scale):
    """
    Recompute G2 from THIS prediction graph.

    Returns number of second-child edges retained.
    """

    scale = np.asarray(
        scale,
        dtype=float,
    )

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(
            attr_keys=[
                "edge_prob",
            ],
            unpack=True,
        )
        .to_pandas()
    )

    remove_ids = []
    forks_kept = 0

    for source_id, group in edges.groupby(
        "source_id"
    ):

        if len(group) != 2:
            continue

        group = group.sort_values(
            ["edge_prob", "edge_id"],
            ascending=[False, True],
        )

        e1 = group.iloc[0]
        e2 = group.iloc[1]

        source_id = int(source_id)
        target1 = int(e1["target_id"])
        target2 = int(e2["target_id"])

        parent = nodes.loc[source_id]
        child1 = nodes.loc[target1]
        child2 = nodes.loc[target2]

        p1_xyz = np.array([
            parent["z"],
            parent["y"],
            parent["x"],
        ], dtype=float) * scale

        c1_xyz = np.array([
            child1["z"],
            child1["y"],
            child1["x"],
        ], dtype=float) * scale

        c2_xyz = np.array([
            child2["z"],
            child2["y"],
            child2["x"],
        ], dtype=float) * scale

        v1 = c1_xyz - p1_xyz
        v2 = c2_xyz - p1_xyz

        d1 = float(
            np.linalg.norm(v1)
        )

        d2 = float(
            np.linalg.norm(v2)
        )

        daughter_sep = float(
            np.linalg.norm(
                c1_xyz - c2_xyz
            )
        )

        if d1 > 0 and d2 > 0:
            cosine = float(
                np.dot(v1, v2)
                / (d1 * d2)
            )

            cosine = np.clip(
                cosine,
                -1.0,
                1.0,
            )

            angle = float(
                np.degrees(
                    np.arccos(cosine)
                )
            )
        else:
            angle = np.nan

        distance_balance = (
            abs(d1 - d2)
            / max(d1 + d2, 1e-12)
        )

        keep_second = (
            float(e2["edge_prob"])
            >= G2_P2_MIN

            and daughter_sep
            >= G2_DAUGHTER_SEP_MIN_UM

            and np.isfinite(angle)

            and angle
            >= G2_ANGLE_MIN_DEG

            and max(d1, d2)
            <= G2_MAX_PARENT_DAUGHTER_UM

            and distance_balance
            <= G2_DISTANCE_BALANCE_MAX
        )

        if keep_second:
            forks_kept += 1
        else:
            remove_ids.append(
                int(e2["edge_id"])
            )

    if remove_ids:
        graph.bulk_remove_edges(
            remove_ids
        )

    return forks_kept


def apply_frozen_component_cleanup(
    graph,
    scale,
):
    """
    Frozen cleanup:
      1. remove singleton components
      2. remove 2-node components whose physical
         displacement exceeds 3.5 µm
    """

    scale = np.asarray(
        scale,
        dtype=float,
    )

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    edge_lookup = {}

    for _, e in edges.iterrows():
        s = int(e["source_id"])
        t = int(e["target_id"])

        edge_lookup[
            frozenset([s, t])
        ] = e

    components = component_sets(
        graph
    )

    remove_nodes = set()

    singleton_count = 0
    long_two_node_count = 0

    for comp in components:

        comp = list(
            map(int, comp)
        )

        # ----------------------------
        # singleton
        # ----------------------------
        if len(comp) == 1:

            remove_nodes.add(
                comp[0]
            )

            singleton_count += 1

        # ----------------------------
        # 2-node tracklet
        # ----------------------------
        elif len(comp) == 2:

            n1, n2 = comp

            key = frozenset(
                [n1, n2]
            )

            if key not in edge_lookup:
                continue

            e = edge_lookup[key]

            source = int(
                e["source_id"]
            )

            target = int(
                e["target_id"]
            )

            a = nodes.loc[source]
            b = nodes.loc[target]

            delta_um = (
                np.array([
                    float(
                        b["z"] - a["z"]
                    ),
                    float(
                        b["y"] - a["y"]
                    ),
                    float(
                        b["x"] - a["x"]
                    ),
                ])
                * scale
            )

            dist_um = float(
                np.linalg.norm(
                    delta_um
                )
            )

            if (
                dist_um
                > TWO_NODE_MAX_DIST_UM
            ):

                remove_nodes.update(
                    [n1, n2]
                )

                long_two_node_count += 1

    if remove_nodes:
        graph.bulk_remove_nodes(
            sorted(remove_nodes)
        )

    return {
        "nodes_removed":
            len(remove_nodes),

        "singletons_removed":
            singleton_count,

        "long_2node_tracklets_removed":
            long_two_node_count,
    }


# =========================================================
# Full evaluation
# =========================================================

records_by_threshold = {}
dataset_rows = []

for det_threshold, pred_dir in (
    PREDICTION_VARIANTS.items()
):

    print("\n" + "=" * 70)
    print(
        f"Evaluating det_threshold={det_threshold}"
    )
    print("=" * 70)

    pred_files = sorted(
        pred_dir.glob("*.geff")
    )

    assert len(pred_files) == 40

    records = []

    for i, pred_path in enumerate(
        pred_files,
        start=1,
    ):

        name = pred_path.stem

        assert name in estimated_map

        pred_g = load_graph(
            pred_path
        )

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        scale = open_dataset(
            TRAIN_DIR / name,
            load_image=False,
        ).scale

        nodes_raw = pred_g.num_nodes()

        # ---------------------------------------------
        # Frozen post-processing
        # ---------------------------------------------

        forks_kept = apply_frozen_g2(
            pred_g,
            scale,
        )

        cleanup = (
            apply_frozen_component_cleanup(
                pred_g,
                scale,
            )
        )

        nodes_final = pred_g.num_nodes()

        # ---------------------------------------------
        # Locked official metric
        # ---------------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=scale,
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        dataset_rows.append({
            "det_threshold":
                det_threshold,

            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "nodes_raw":
                nodes_raw,

            "nodes_final":
                nodes_final,

            "nodes_removed":
                cleanup["nodes_removed"],

            "forks_kept":
                forks_kept,

            "singletons_removed":
                cleanup[
                    "singletons_removed"
                ],

            "long_2node_removed":
                cleanup[
                    "long_2node_tracklets_removed"
                ],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],
        })

        if i % 10 == 0:
            print(
                f"{i}/40"
            )

    records_by_threshold[
        det_threshold
    ] = records


# =========================================================
# Fold summaries
# =========================================================

summary_rows = []

for det_threshold, records in (
    records_by_threshold.items()
):

    s = summarise(
        records
    )

    df_t = pd.DataFrame(
        [
            r
            for r in dataset_rows
            if r["det_threshold"]
            == det_threshold
        ]
    )

    summary_rows.append({
        "det_threshold":
            det_threshold,

        "raw_pred_nodes":
            int(
                df_t[
                    "nodes_raw"
                ].sum()
            ),

        "final_pred_nodes":
            int(
                df_t[
                    "nodes_final"
                ].sum()
            ),

        "nodes_removed":
            int(
                df_t[
                    "nodes_removed"
                ].sum()
            ),

        "forks_kept":
            int(
                df_t[
                    "forks_kept"
                ].sum()
            ),

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


threshold_comparison = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "det_threshold"
    )
    .reset_index(drop=True)
)

print(
    "\n=== DETECTION THRESHOLD COMPARISON ==="
)

display(
    threshold_comparison
)


# =========================================================
# Reproduction check for existing 0.99 policy
# =========================================================

EXPECTED_099 = 0.738776

score_099 = float(
    threshold_comparison.loc[
        threshold_comparison[
            "det_threshold"
        ] == 0.99,
        "score",
    ].iloc[0]
)

print(
    "\n0.99 frozen-policy score:",
    score_099,
)

print(
    "Difference vs previous:",
    score_099 - EXPECTED_099,
)

0.741854 @ det_threshold=0.995​

In [ ]:
# =========================================================
# Detection threshold comparison by prefix
# =========================================================

prefix_rows = []

for det_threshold, pred_dir in PREDICTION_VARIANTS.items():

    names = sorted(
        p.stem
        for p in pred_dir.glob("*.geff")
    )

    records = records_by_threshold[
        det_threshold
    ]

    assert len(names) == len(records) == 40

    for prefix in ["44b6", "6bba"]:

        prefix_records = [
            m
            for name, m in zip(
                names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        s = summarise(
            prefix_records
        )

        prefix_node_ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        prefix_rows.append({
            "det_threshold":
                det_threshold,

            "prefix":
                prefix,

            "n":
                s["n"],

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "division_tp":
                s["division_tp"],

            "division_fp":
                s["division_fp"],

            "division_fn":
                s["division_fn"],

            "node_recall":
                s["node_recall"],

            "median_node_ratio":
                float(
                    np.median(
                        prefix_node_ratios
                    )
                ),

            "score":
                s["score"],
        })


threshold_by_prefix = (
    pd.DataFrame(prefix_rows)
    .sort_values(
        ["prefix", "det_threshold"]
    )
    .reset_index(drop=True)
)

print(
    "=== DETECTION THRESHOLD BY PREFIX ==="
)

display(
    threshold_by_prefix
)


print(
    "\n=== SCORE PIVOT ==="
)

display(
    threshold_by_prefix.pivot(
        index="prefix",
        columns="det_threshold",
        values="score",
    )
)


print(
    "\n=== NODE RECALL PIVOT ==="
)

display(
    threshold_by_prefix.pivot(
        index="prefix",
        columns="det_threshold",
        values="node_recall",
    )
)


print(
    "\n=== MEDIAN NODE RATIO PIVOT ==="
)

display(
    threshold_by_prefix.pivot(
        index="prefix",
        columns="det_threshold",
        values="median_node_ratio",
    )
)

0.995 在两个 domain 上都提高了最终 score：

44b6: 0.742935 → 0.746337

6bba: 0.737818 → 0.740809

全局最佳仍然是：det threshold = 0.995, Fold0 = 0.741854
	​


In [ ]:
threshold = 0.9975
method = "official_baseline_fold0_det09975"

cmd = [
    sys.executable,
    str(PREDICT_SCRIPT),

    "--data-dir",
    str(TRAIN_DIR),

    "--splits",
    str(SPLITS),

    "--split",
    "0",

    "--method",
    method,

    "--weights",
    str(CKPT),

    "--det-threshold",
    str(threshold),
]

start = time.time()

process = subprocess.Popen(
    cmd,
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
wall = time.time() - start

print("\nReturn code:", return_code)
print(
    f"Wall time: {wall:.1f}s = {wall / 60:.2f} min"
)

assert return_code == 0

pred_dir_09975 = (
    PROJECT
    / "predictions"
    / "markdd520"
    / method
    / "split_0"
)

pred_files = sorted(
    pred_dir_09975.glob("*.geff")
)

print("Predicted GEFFs:", len(pred_files))

assert len(pred_files) == 40

print("\n0.9975 inference: OK")

In [ ]:
# =========================================================
# Evaluate only det_threshold = 0.9975
# Frozen:
#   G2 division policy
#   singleton removal
#   >3.5 µm 2-node tracklet removal
# =========================================================

DET_09975_DIR = (
    PROJECT
    / "predictions"
    / "markdd520"
    / "official_baseline_fold0_det09975"
    / "split_0"
)

pred_files_09975 = sorted(
    DET_09975_DIR.glob("*.geff")
)

assert len(pred_files_09975) == 40

records_09975 = []
dataset_rows_09975 = []

for i, pred_path in enumerate(
    pred_files_09975,
    start=1,
):
    name = pred_path.stem

    pred_g = load_graph(
        pred_path
    )

    gt_g = load_graph(
        TRAIN_DIR / f"{name}.geff"
    )

    scale = open_dataset(
        TRAIN_DIR / name,
        load_image=False,
    ).scale

    nodes_raw = pred_g.num_nodes()

    # ---------------------------------------------
    # Same frozen G2
    # ---------------------------------------------
    forks_kept = apply_frozen_g2(
        pred_g,
        scale,
    )

    # ---------------------------------------------
    # Same frozen component cleanup
    # ---------------------------------------------
    cleanup = apply_frozen_component_cleanup(
        pred_g,
        scale,
    )

    nodes_final = pred_g.num_nodes()

    # ---------------------------------------------
    # Official metric
    # ---------------------------------------------
    er = official_evaluate(
        pred_g,
        gt_g,
        scale=scale,
        max_distance=7.0,
    )

    recall = node_recall(
        pred_g,
        gt_g,
    )

    m = per_sample_metrics(
        er,
        float(estimated_map[name]),
        recall,
    )

    records_09975.append(m)

    dataset_rows_09975.append({
        "dataset": name,
        "prefix": prefix_map[name],

        "nodes_raw": nodes_raw,
        "nodes_final": nodes_final,
        "nodes_removed":
            cleanup["nodes_removed"],

        "forks_kept":
            forks_kept,

        "node_recall":
            m["node_recall"],

        "node_ratio":
            m["total_node_ratio"],

        "edge_jaccard":
            m["edge_jaccard"],

        "adj_edge_jaccard":
            m["adj_edge_jaccard"],
    })

    if i % 10 == 0:
        print(f"Processed {i}/40")


# =========================================================
# Overall 0.9975 result
# =========================================================

summary_09975 = summarise(
    records_09975
)

df_09975 = pd.DataFrame(
    dataset_rows_09975
)

print(
    "\n=== DET THRESHOLD 0.9975 OVERALL ==="
)

for k, v in summary_09975.items():
    print(f"{k}: {v}")

print(
    "raw_pred_nodes:",
    df_09975["nodes_raw"].sum(),
)

print(
    "final_pred_nodes:",
    df_09975["nodes_final"].sum(),
)

print(
    "median_node_ratio:",
    np.median([
        m["total_node_ratio"]
        for m in records_09975
    ]),
)


# =========================================================
# Prefix decomposition
# =========================================================

names_09975 = [
    p.stem
    for p in pred_files_09975
]

prefix_rows_09975 = []

for prefix in ["44b6", "6bba"]:

    recs = [
        m
        for name, m in zip(
            names_09975,
            records_09975,
        )
        if prefix_map[name] == prefix
    ]

    s = summarise(recs)

    ratios = [
        m["total_node_ratio"]
        for name, m in zip(
            names_09975,
            records_09975,
        )
        if prefix_map[name] == prefix
    ]

    prefix_rows_09975.append({
        "prefix": prefix,
        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(np.median(ratios)),

        "score":
            s["score"],
    })


prefix_09975 = pd.DataFrame(
    prefix_rows_09975
)

print(
    "\n=== 0.9975 BY PREFIX ==="
)

display(prefix_09975)


# =========================================================
# Exact domain-aware candidate:
#
# 44b6 -> 0.9975
# 6bba -> 0.9950
# =========================================================

names_0995 = sorted(
    p.stem
    for p in PREDICTION_VARIANTS[
        0.995
    ].glob("*.geff")
)

records_0995 = (
    records_by_threshold[
        0.995
    ]
)

assert (
    names_0995
    == names_09975
)

record_map_0995 = dict(
    zip(
        names_0995,
        records_0995,
    )
)

record_map_09975 = dict(
    zip(
        names_09975,
        records_09975,
    )
)

mixed_records = []

for name in names_09975:

    if prefix_map[name] == "44b6":
        mixed_records.append(
            record_map_09975[name]
        )
    else:
        mixed_records.append(
            record_map_0995[name]
        )


mixed_summary = summarise(
    mixed_records
)

print(
    "\n=== DOMAIN-AWARE CANDIDATE ==="
)

print(
    "44b6 det_threshold = 0.9975"
)
print(
    "6bba det_threshold = 0.9950"
)

for k, v in mixed_summary.items():
    print(f"{k}: {v}")


GLOBAL_0995_SCORE = 0.741854

print(
    "\nDelta vs global 0.995:",
    mixed_summary["score"]
    - GLOBAL_0995_SCORE,
)